Cell 1：环境检查

In [1]:
# ============================================================
# Cell 1
# Environment check
# ============================================================

import sys
import torch
import numpy as np
import sklearn
import yaml


print("=" * 80)
print("KAGGLE ENVIRONMENT")
print("=" * 80)

print("Python:", sys.version)
print("Python executable:", sys.executable)
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("CUDA available:", torch.cuda.is_available())


if not torch.cuda.is_available():
    raise RuntimeError(
        "没有检测到 GPU，请先在 Kaggle Settings 中开启 GPU。"
    )


print("GPU:", torch.cuda.get_device_name(0))

print("\nPASS: Kaggle environment ready.")

KAGGLE ENVIRONMENT
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Python executable: /usr/bin/python3
PyTorch: 2.10.0+cu128
NumPy: 2.0.2
scikit-learn: 1.6.1
CUDA available: True
GPU: Tesla T4

PASS: Kaggle environment ready.


Cell 2：准备 anndata

In [2]:
# ============================================================
# Cell 2
# anndata
# ============================================================

import sys
import subprocess
import importlib.util


if importlib.util.find_spec("anndata") is None:

    print("Installing anndata...")

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--no-deps",
            "anndata==0.11.4",
        ],
        check=True,
    )


import anndata


print("anndata:", anndata.__version__)
print("NumPy:", __import__("numpy").__version__)

print("\nPASS: anndata ready.")

Installing anndata...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 9.0 MB/s eta 0:00:00
anndata: 0.11.4
NumPy: 2.0.2

PASS: anndata ready.


Cell 3：重新克隆干净 SpaMGCL

In [3]:
# ============================================================
# Cell 3
# Clone clean SpaMGCL repository
# ============================================================

from pathlib import Path
import shutil
import subprocess


REPO_ROOT = Path(
    "/kaggle/working/SpaMGCL"
)

PROJECT_ROOT = (
    REPO_ROOT
    / "SpaMGCL"
)


if REPO_ROOT.exists():

    print("Removing existing repository:")
    print(REPO_ROOT)

    shutil.rmtree(
        REPO_ROOT
    )


subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "--branch",
        "main",
        "https://github.com/huqian122/SpaMGCL.git",
        str(REPO_ROOT),
    ],
    check=True,
)


assert PROJECT_ROOT.exists(), (
    f"Project root not found: {PROJECT_ROOT}"
)


print("\nProject root:")
print(PROJECT_ROOT)


print("\nRepository commit:")

subprocess.run(
    [
        "git",
        "-C",
        str(REPO_ROOT),
        "log",
        "-1",
        "--oneline",
    ],
    check=True,
)


print(
    "\nPASS: clean repository cloned."
)

Cloning into '/kaggle/working/SpaMGCL'...



Project root:
/kaggle/working/SpaMGCL/SpaMGCL

Repository commit:
b4abd3d Add files via upload

PASS: clean repository cloned.


Cell 4：检查源码 + 两个正式 base config

In [5]:
# ============================================================
# Cell 4
# Audit source code and ablation base configs
# Dataset-specific full-model loss weights are preserved
# ============================================================

from pathlib import Path
import os
import yaml
import subprocess
import sys


PROJECT_ROOT = Path(
    "/kaggle/working/SpaMGCL/SpaMGCL"
)

os.chdir(PROJECT_ROOT)


ABLATION_BASE_CONFIGS = {
    "hlna1":
        PROJECT_ROOT
        / "configs/final_clean/hlna1_clean_200.yaml",

    "e185":
        PROJECT_ROOT
        / "configs/final_clean/e185_clean_200.yaml",
}


EXPECTED_DATASETS = {
    "hlna1": "HLN-A1",
    "e185": "E18.5",
}


EXPECTED_CLUSTERS = {
    "hlna1": 10,
    "e185": 14,
}


# Important:
# preserve dataset-specific Full-model loss weights
EXPECTED_FULL_LOSS = {
    "hlna1": {
        "lambda_rec": 1.0,
        "lambda_mgcl": 1.0,
        "lambda_cluster": 0.1,
        "lambda_spatial": 0.0,
    },

    "e185": {
        "lambda_rec": 1.0,
        "lambda_mgcl": 3.0,
        "lambda_cluster": 0.1,
        "lambda_spatial": 0.0,
    },
}


REQUIRED_CODE = [
    PROJECT_ROOT / "src/clustering/refinement.py",
    PROJECT_ROOT / "src/clustering/predict.py",
    PROJECT_ROOT / "experiments/run_exp.py",
    PROJECT_ROOT / "scripts/audit_run.py",
    PROJECT_ROOT / "tests/test_bsrr.py",
]


print("=" * 100)
print("SOURCE CODE CHECK")
print("=" * 100)


for path in REQUIRED_CODE:

    assert path.exists(), (
        f"Missing source file: {path}"
    )

    print(
        "PASS:",
        path.relative_to(PROJECT_ROOT)
    )


subprocess.run(
    [
        sys.executable,
        "-m",
        "py_compile",
        str(PROJECT_ROOT / "experiments/run_exp.py"),
        str(PROJECT_ROOT / "src/clustering/refinement.py"),
        str(PROJECT_ROOT / "src/clustering/predict.py"),
        str(PROJECT_ROOT / "scripts/audit_run.py"),
    ],
    check=True,
)


print("\nPASS: Python syntax check.")


print("\n" + "=" * 100)
print("ABLATION BASE CONFIG AUDIT")
print("=" * 100)


for key, config_path in ABLATION_BASE_CONFIGS.items():

    assert config_path.exists(), (
        f"Missing base config: {config_path}"
    )

    with config_path.open(
        "r",
        encoding="utf-8",
    ) as f:
        cfg = yaml.safe_load(f)


    dataset = cfg["experiment"]["dataset"]
    warm_up = int(cfg["training"]["warm_up_epochs"])
    n_clusters = int(cfg["clustering"]["n_clusters"])
    loss_cfg = cfg["loss"]

    expected_loss = EXPECTED_FULL_LOSS[key]


    print(f"\n{key}")

    print(f"  dataset          = {dataset}")
    print(f"  warm_up_epochs   = {warm_up}")
    print(f"  clusters         = {n_clusters}")
    print(f"  embedding        = {cfg['clustering']['embedding']}")
    print(f"  KMeans n_init    = {cfg['clustering']['n_init']}")
    print(f"  KMeans rs        = {cfg['clustering']['random_state']}")
    print(f"  BSRR             = {cfg['refinement']['enabled']}")
    print(f"  BSRR spatial_k   = {cfg['refinement']['spatial_k']}")
    print(f"  lambda_rec       = {loss_cfg['lambda_rec']}")
    print(f"  lambda_mgcl      = {loss_cfg['lambda_mgcl']}")
    print(f"  lambda_cluster   = {loss_cfg['lambda_cluster']}")
    print(f"  lambda_spatial   = {loss_cfg['lambda_spatial']}")


    assert dataset == EXPECTED_DATASETS[key]
    assert warm_up == 10
    assert n_clusters == EXPECTED_CLUSTERS[key]

    assert cfg["clustering"]["method"].lower() == "kmeans"
    assert cfg["clustering"]["embedding"].lower() == "concat_z"

    assert int(cfg["clustering"]["n_init"]) == 20
    assert int(cfg["clustering"]["random_state"]) == 0

    assert cfg["refinement"]["enabled"] is True
    assert cfg["refinement"]["method"].lower() == "bsrr"
    assert int(cfg["refinement"]["spatial_k"]) == 3

    assert (
        cfg["evaluation"]["nmi_average_method"]
        == "max"
    )


    for loss_name, expected_value in expected_loss.items():

        actual_value = float(
            loss_cfg[loss_name]
        )

        assert actual_value == expected_value, (
            f"{key}: {loss_name} "
            f"{actual_value} != {expected_value}"
        )


print(
    "\nPASS: dataset-specific Full-model "
    "loss weights verified."
)

print("\nFrozen Full loss protocol:")

for key, loss_cfg in EXPECTED_FULL_LOSS.items():

    print(
        f"{key:7s} | "
        f"rec={loss_cfg['lambda_rec']} | "
        f"mgcl={loss_cfg['lambda_mgcl']} | "
        f"cluster={loss_cfg['lambda_cluster']} | "
        f"spatial={loss_cfg['lambda_spatial']}"
    )

SOURCE CODE CHECK
PASS: src/clustering/refinement.py
PASS: src/clustering/predict.py
PASS: experiments/run_exp.py
PASS: scripts/audit_run.py
PASS: tests/test_bsrr.py

PASS: Python syntax check.

ABLATION BASE CONFIG AUDIT

hlna1
  dataset          = HLN-A1
  warm_up_epochs   = 10
  clusters         = 10
  embedding        = concat_z
  KMeans n_init    = 20
  KMeans rs        = 0
  BSRR             = True
  BSRR spatial_k   = 3
  lambda_rec       = 1.0
  lambda_mgcl      = 1.0
  lambda_cluster   = 0.1
  lambda_spatial   = 0.0

e185
  dataset          = E18.5
  warm_up_epochs   = 10
  clusters         = 14
  embedding        = concat_z
  KMeans n_init    = 20
  KMeans rs        = 0
  BSRR             = True
  BSRR spatial_k   = 3
  lambda_rec       = 1.0
  lambda_mgcl      = 3.0
  lambda_cluster   = 0.1
  lambda_spatial   = 0.0

PASS: dataset-specific Full-model loss weights verified.

Frozen Full loss protocol:
hlna1   | rec=1.0 | mgcl=1.0 | cluster=0.1 | spatial=0.0
e185    | rec=1.0 |

Cell 5：检查数据集

In [6]:
# ============================================================
# Cell 5
# Dataset mount check
# ============================================================

from pathlib import Path


DATA_ROOT = Path(
    "/kaggle/input/datasets/wuvdji/smgc-data"
)


print("=" * 80)
print("DATASET CHECK")
print("=" * 80)

print(
    "Data root:",
    DATA_ROOT,
)

print(
    "Exists:",
    DATA_ROOT.exists(),
)


assert DATA_ROOT.exists(), (
    "没有找到 smgc-data。\n"
    "请先在 Kaggle Notebook 中 Add Input。"
)


expected_dirs = [

    DATA_ROOT
    / "Human_Lymph_Nodes",

    DATA_ROOT
    / "E18.5_mouse_brain",
]


for path in expected_dirs:

    print(
        path.name,
        "->",
        path.exists(),
    )

    assert path.exists(), (
        f"Missing dataset directory: "
        f"{path}"
    )


print(
    "\nPASS: HLN-A1 and E18.5 datasets mounted."
)

DATASET CHECK
Data root: /kaggle/input/datasets/wuvdji/smgc-data
Exists: True
Human_Lymph_Nodes -> True
E18.5_mouse_brain -> True

PASS: HLN-A1 and E18.5 datasets mounted.


Cell 6：生成正式消融 20 个配置

In [7]:
# ============================================================
# Cell 6
# Generate formal ablation configs
#
# IMPORTANT:
# Start from each dataset's own Full config.
# Change ONLY ONE target loss.
# ============================================================

from pathlib import Path
import copy
import yaml


PROJECT_ROOT = Path(
    "/kaggle/working/SpaMGCL/SpaMGCL"
)

DATA_ROOT = Path(
    "/kaggle/input/datasets/wuvdji/smgc-data"
)


ABLATION_CONFIG_DIR = (
    PROJECT_ROOT
    / "configs"
    / "ablation_400ep"
)

ABLATION_CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


ABLATION_RESULT_ROOT_NAME = (
    "results_ablation_400"
)


ABLATION_SEEDS = [
    0,
    1,
    2,
    3,
    4,
]


ABLATION_VARIANTS = [
    "wo_sample",
    "wo_cluster",
]


ABLATION_CONFIGS = []


for dataset_key, base_path in (
    ABLATION_BASE_CONFIGS.items()
):

    with base_path.open(
        "r",
        encoding="utf-8",
    ) as f:

        base_cfg = yaml.safe_load(f)


    # --------------------------------------------------------
    # Remember the original Full-model loss weights
    # --------------------------------------------------------

    original_loss = copy.deepcopy(
        base_cfg["loss"]
    )


    for variant_name in ABLATION_VARIANTS:

        for seed in ABLATION_SEEDS:

            cfg = copy.deepcopy(
                base_cfg
            )


            experiment_name = (
                f"{dataset_key}_"
                f"ablation_"
                f"{variant_name}_"
                f"400ep_"
                f"seed{seed}"
            )


            # ====================================================
            # Experiment
            # ====================================================

            cfg["experiment"]["name"] = (
                experiment_name
            )

            cfg["experiment"]["dataset"] = (
                EXPECTED_DATASETS[
                    dataset_key
                ]
            )

            cfg["experiment"]["seed"] = seed


            if (
                "epochs"
                in cfg["experiment"]
            ):

                cfg["experiment"]["epochs"] = 400


            # ====================================================
            # Training
            # ====================================================

            cfg["training"]["epochs"] = 400


            if (
                "seed"
                in cfg["training"]
            ):

                cfg["training"]["seed"] = seed


            cfg["training"].pop(
                "resume_from",
                None,
            )


            # ====================================================
            # Data
            # ====================================================

            cfg["data"]["root"] = str(
                DATA_ROOT
            )


            # ====================================================
            # Output
            # ====================================================

            cfg["output"]["root"] = (
                ABLATION_RESULT_ROOT_NAME
            )


            # ====================================================
            # Frozen official readout
            # ====================================================

            cfg["clustering"]["method"] = (
                "kmeans"
            )

            cfg["clustering"]["embedding"] = (
                "concat_z"
            )

            cfg["clustering"]["n_clusters"] = (
                EXPECTED_CLUSTERS[
                    dataset_key
                ]
            )

            cfg["clustering"]["n_init"] = 20
            cfg["clustering"]["random_state"] = 0


            cfg["refinement"]["enabled"] = True
            cfg["refinement"]["method"] = "bsrr"
            cfg["refinement"]["spatial_k"] = 3


            cfg[
                "evaluation"
            ][
                "nmi_average_method"
            ] = "max"


            # ====================================================
            # Reset to dataset-specific Full loss first
            # ====================================================

            cfg["loss"] = copy.deepcopy(
                original_loss
            )


            # ====================================================
            # SINGLE-FACTOR ABLATION
            # ====================================================

            if variant_name == "wo_sample":

                # Only remove sample-level contrastive loss
                cfg["loss"]["lambda_mgcl"] = 0.0


            elif variant_name == "wo_cluster":

                # Only remove cluster-level contrastive loss
                cfg["loss"]["lambda_cluster"] = 0.0


            else:

                raise RuntimeError(
                    f"Unknown variant: {variant_name}"
                )


            # ====================================================
            # Save
            # ====================================================

            config_path = (
                ABLATION_CONFIG_DIR
                / f"{experiment_name}.yaml"
            )


            with config_path.open(
                "w",
                encoding="utf-8",
            ) as f:

                yaml.safe_dump(
                    cfg,
                    f,
                    sort_keys=False,
                    allow_unicode=True,
                )


            ABLATION_CONFIGS.append(
                config_path
            )


assert len(
    ABLATION_CONFIGS
) == 20


print("=" * 100)
print("FORMAL ABLATION CONFIG GENERATION")
print("=" * 100)


for dataset_key in [
    "hlna1",
    "e185",
]:

    for variant_name in [
        "wo_sample",
        "wo_cluster",
    ]:

        paths = [
            p
            for p in ABLATION_CONFIGS
            if p.name.startswith(
                f"{dataset_key}_"
                f"ablation_"
                f"{variant_name}_"
            )
        ]


        assert len(paths) == 5


        print(
            f"{dataset_key:7s} | "
            f"{variant_name:10s} | "
            f"{len(paths)} configs"
        )


print(
    "\nTotal configs:",
    len(ABLATION_CONFIGS),
)

print(
    "Config directory:",
    ABLATION_CONFIG_DIR,
)


print(
    "\nPASS: 20 single-factor "
    "formal ablation configs generated."
)

FORMAL ABLATION CONFIG GENERATION
hlna1   | wo_sample  | 5 configs
hlna1   | wo_cluster | 5 configs
e185    | wo_sample  | 5 configs
e185    | wo_cluster | 5 configs

Total configs: 20
Config directory: /kaggle/working/SpaMGCL/SpaMGCL/configs/ablation_400ep

PASS: 20 single-factor formal ablation configs generated.


Cell 7 核查全部 20 组正式消融实验配置

In [8]:
# ============================================================
# Cell 7
# Audit all 20 formal ablation configs
# ============================================================

import pandas as pd
import yaml


audit_rows = []


for config_path in sorted(
    ABLATION_CONFIGS
):

    with config_path.open(
        "r",
        encoding="utf-8",
    ) as f:

        cfg = yaml.safe_load(f)


    name = cfg["experiment"]["name"]
    dataset = cfg["experiment"]["dataset"]
    seed = int(cfg["experiment"]["seed"])


    if "_wo_sample_" in name:

        variant = "wo_sample"

    elif "_wo_cluster_" in name:

        variant = "wo_cluster"

    else:

        raise RuntimeError(
            f"Unknown variant: {name}"
        )


    audit_rows.append(
        {
            "dataset":
                dataset,

            "variant":
                variant,

            "seed":
                seed,

            "epochs":
                int(
                    cfg["training"]["epochs"]
                ),

            "warm_up":
                int(
                    cfg["training"][
                        "warm_up_epochs"
                    ]
                ),

            "lambda_rec":
                float(
                    cfg["loss"]["lambda_rec"]
                ),

            "lambda_mgcl":
                float(
                    cfg["loss"]["lambda_mgcl"]
                ),

            "lambda_cluster":
                float(
                    cfg["loss"]["lambda_cluster"]
                ),

            "lambda_spatial":
                float(
                    cfg["loss"]["lambda_spatial"]
                ),

            "K":
                int(
                    cfg["clustering"][
                        "n_clusters"
                    ]
                ),

            "embedding":
                cfg["clustering"][
                    "embedding"
                ],

            "n_init":
                int(
                    cfg["clustering"][
                        "n_init"
                    ]
                ),

            "kmeans_rs":
                int(
                    cfg["clustering"][
                        "random_state"
                    ]
                ),

            "BSRR":
                cfg["refinement"][
                    "enabled"
                ],

            "spatial_k":
                int(
                    cfg["refinement"][
                        "spatial_k"
                    ]
                ),

            "NMI":
                cfg["evaluation"][
                    "nmi_average_method"
                ],
        }
    )


audit_df = pd.DataFrame(
    audit_rows
)


print(
    audit_df.to_string(
        index=False
    )
)


assert len(audit_df) == 20

assert (
    audit_df["epochs"] == 400
).all()

assert (
    audit_df["warm_up"] == 10
).all()

assert (
    audit_df["lambda_rec"] == 1.0
).all()

assert (
    audit_df["lambda_spatial"] == 0.0
).all()

assert (
    audit_df["embedding"] == "concat_z"
).all()

assert (
    audit_df["n_init"] == 20
).all()

assert (
    audit_df["kmeans_rs"] == 0
).all()

assert (
    audit_df["BSRR"] == True
).all()

assert (
    audit_df["spatial_k"] == 3
).all()

assert (
    audit_df["NMI"] == "max"
).all()


# ============================================================
# Dataset-specific single-factor checks
# ============================================================

for dataset, full_mgcl, expected_k in [

    ("HLN-A1", 1.0, 10),

    ("E18.5", 3.0, 14),
]:

    # --------------------------------------------------------
    # w/o Sample
    # --------------------------------------------------------

    d = audit_df[
        (audit_df["dataset"] == dataset)
        &
        (audit_df["variant"] == "wo_sample")
    ]


    assert len(d) == 5

    assert (
        sorted(d["seed"].tolist())
        == [0, 1, 2, 3, 4]
    )

    assert (
        d["lambda_mgcl"] == 0.0
    ).all()

    assert (
        d["lambda_cluster"] == 0.1
    ).all()

    assert (
        d["K"] == expected_k
    ).all()


    # --------------------------------------------------------
    # w/o Cluster
    # --------------------------------------------------------

    d = audit_df[
        (audit_df["dataset"] == dataset)
        &
        (audit_df["variant"] == "wo_cluster")
    ]


    assert len(d) == 5

    assert (
        sorted(d["seed"].tolist())
        == [0, 1, 2, 3, 4]
    )

    # Critical:
    # sample-level loss remains at Full value
    assert (
        d["lambda_mgcl"] == full_mgcl
    ).all()

    assert (
        d["lambda_cluster"] == 0.0
    ).all()

    assert (
        d["K"] == expected_k
    ).all()


print(
    "\n" + "=" * 100
)

print(
    "PASS: FORMAL ABLATION "
    "PROTOCOL FROZEN"
)

print(
    "=" * 100
)


print(
    "\nHLN-A1 Full loss:"
)

print(
    "rec=1.0 | mgcl=1.0 | "
    "cluster=0.1 | spatial=0.0"
)


print(
    "\nE18.5 Full loss:"
)

print(
    "rec=1.0 | mgcl=3.0 | "
    "cluster=0.1 | spatial=0.0"
)


print(
    "\nwo_sample:"
)

print(
    "Only lambda_mgcl -> 0"
)


print(
    "\nwo_cluster:"
)

print(
    "Only lambda_cluster -> 0"
)


print(
    "\n20 configs verified."
)

print(
    "DO NOT START TRAINING YET."
)

dataset    variant  seed  epochs  warm_up  lambda_rec  lambda_mgcl  lambda_cluster  lambda_spatial  K embedding  n_init  kmeans_rs  BSRR  spatial_k NMI
  E18.5 wo_cluster     0     400       10         1.0          3.0             0.0             0.0 14  concat_z      20          0  True          3 max
  E18.5 wo_cluster     1     400       10         1.0          3.0             0.0             0.0 14  concat_z      20          0  True          3 max
  E18.5 wo_cluster     2     400       10         1.0          3.0             0.0             0.0 14  concat_z      20          0  True          3 max
  E18.5 wo_cluster     3     400       10         1.0          3.0             0.0             0.0 14  concat_z      20          0  True          3 max
  E18.5 wo_cluster     4     400       10         1.0          3.0             0.0             0.0 14  concat_z      20          0  True          3 max
  E18.5  wo_sample     0     400       10         1.0          0.0             0.1      

Cell 8：建立独立 ablation runner

In [9]:
# ============================================================
# Cell 8
# Create isolated ablation runner
#
# IMPORTANT:
# The training code is NOT modified.
# All ablation changes come from YAML configs only.
# ============================================================

from pathlib import Path
import shutil
import hashlib
import subprocess
import sys


PROJECT_ROOT = Path(
    "/kaggle/working/SpaMGCL/SpaMGCL"
)

ORIGINAL_RUNNER = (
    PROJECT_ROOT
    / "experiments"
    / "run_exp.py"
)

ABLATION_RUNNER = (
    PROJECT_ROOT
    / "experiments"
    / "run_exp_ablation_400.py"
)


assert ORIGINAL_RUNNER.exists(), (
    f"Missing original runner: {ORIGINAL_RUNNER}"
)


# ============================================================
# Copy original runner byte-for-byte
# ============================================================

shutil.copy2(
    ORIGINAL_RUNNER,
    ABLATION_RUNNER,
)


assert ABLATION_RUNNER.exists()


# ============================================================
# SHA256
# ============================================================

def sha256_file(path):

    h = hashlib.sha256()

    with Path(path).open("rb") as f:

        while True:

            chunk = f.read(
                1024 * 1024
            )

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


original_sha = sha256_file(
    ORIGINAL_RUNNER
)

ablation_sha = sha256_file(
    ABLATION_RUNNER
)


print("=" * 90)
print("ABLATION RUNNER")
print("=" * 90)

print(
    "Original:",
    ORIGINAL_RUNNER
)

print(
    "Ablation:",
    ABLATION_RUNNER
)

print(
    "\nOriginal SHA256:",
    original_sha
)

print(
    "Ablation SHA256:",
    ablation_sha
)


assert original_sha == ablation_sha, (
    "Ablation runner is not identical "
    "to original run_exp.py"
)


# ============================================================
# Syntax check
# ============================================================

subprocess.run(
    [
        sys.executable,
        "-m",
        "py_compile",
        str(ABLATION_RUNNER),
    ],
    check=True,
)


print(
    "\nPASS: ablation runner is "
    "byte-identical to run_exp.py."
)

print(
    "PASS: Python syntax check."
)

print(
    "\nAll ablation differences "
    "come ONLY from config files."
)

ABLATION RUNNER
Original: /kaggle/working/SpaMGCL/SpaMGCL/experiments/run_exp.py
Ablation: /kaggle/working/SpaMGCL/SpaMGCL/experiments/run_exp_ablation_400.py

Original SHA256: c358f8307e3aa492eaff3c9227fed37e9cf6f7fad75b26587f6b60c417c00bdc
Ablation SHA256: c358f8307e3aa492eaff3c9227fed37e9cf6f7fad75b26587f6b60c417c00bdc

PASS: ablation runner is byte-identical to run_exp.py.
PASS: Python syntax check.

All ablation differences come ONLY from config files.


Cell 9：定义正式消融运行与审计函数

In [10]:
# ============================================================
# Cell 9
# Formal ablation run + audit helper
# ============================================================

from pathlib import Path
import json
import yaml
import subprocess
import sys
import numpy as np

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)


PROJECT_ROOT = Path(
    "/kaggle/working/SpaMGCL/SpaMGCL"
)

ABLATION_RESULT_ROOT = (
    PROJECT_ROOT
    / "results_ablation_400"
)

ABLATION_RESULT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# Files required from a COMPLETED ablation run
ABLATION_REQUIRED_FILES = [
    "metrics.json",
    "config.yaml",
    "manifest.json",

    "gt_labels.npy",
    "coords.npy",

    "pred_labels.npy",
    "pred_concat_z_kmeans.npy",
    "pred_q_argmax.npy",

    "z_concat.npy",
    "z_bsrr.npy",

    "checkpoint_last.pt",
]


def expected_ablation_loss(
    dataset,
    variant,
):
    """
    Return exact expected loss weights for one
    dataset × ablation variant.
    """

    full_mgcl = {
        "HLN-A1": 1.0,
        "E18.5": 3.0,
    }[dataset]


    if variant == "wo_sample":

        return {
            "lambda_rec": 1.0,
            "lambda_mgcl": 0.0,
            "lambda_cluster": 0.1,
            "lambda_spatial": 0.0,
        }


    elif variant == "wo_cluster":

        return {
            "lambda_rec": 1.0,
            "lambda_mgcl": full_mgcl,
            "lambda_cluster": 0.0,
            "lambda_spatial": 0.0,
        }


    else:

        raise ValueError(
            f"Unknown variant: {variant}"
        )


def infer_ablation_variant(name):

    if "_wo_sample_" in name:
        return "wo_sample"

    if "_wo_cluster_" in name:
        return "wo_cluster"

    raise ValueError(
        f"Cannot infer variant from {name}"
    )


def audit_ablation_output(
    config_path,
    run_dir,
):

    config_path = Path(
        config_path
    )

    run_dir = Path(
        run_dir
    )


    # ========================================================
    # Load requested config
    # ========================================================

    with config_path.open(
        "r",
        encoding="utf-8",
    ) as f:

        requested_cfg = yaml.safe_load(f)


    exp = requested_cfg[
        "experiment"
    ]

    dataset = exp["dataset"]
    seed = int(exp["seed"])
    name = exp["name"]

    variant = infer_ablation_variant(
        name
    )


    # ========================================================
    # File completeness
    # ========================================================

    missing = [
        filename
        for filename in ABLATION_REQUIRED_FILES
        if not (
            run_dir
            / filename
        ).exists()
    ]


    if missing:

        raise RuntimeError(
            f"{name}: completed run "
            f"is missing files:\n{missing}"
        )


    # ========================================================
    # Load actual saved config
    # ========================================================

    with (
        run_dir
        / "config.yaml"
    ).open(
        "r",
        encoding="utf-8",
    ) as f:

        saved_cfg = yaml.safe_load(f)


    # ========================================================
    # Protocol check
    # ========================================================

    assert (
        saved_cfg[
            "experiment"
        ]["dataset"]
        == dataset
    )

    assert int(
        saved_cfg[
            "experiment"
        ]["seed"]
    ) == seed

    assert int(
        saved_cfg[
            "training"
        ]["epochs"]
    ) == 400

    assert int(
        saved_cfg[
            "training"
        ]["warm_up_epochs"]
    ) == 10


    assert (
        saved_cfg[
            "clustering"
        ]["embedding"]
        == "concat_z"
    )

    assert (
        saved_cfg[
            "clustering"
        ]["method"]
        == "kmeans"
    )

    assert int(
        saved_cfg[
            "clustering"
        ]["n_init"]
    ) == 20

    assert int(
        saved_cfg[
            "clustering"
        ]["random_state"]
    ) == 0


    assert (
        saved_cfg[
            "refinement"
        ]["enabled"]
        is True
    )

    assert (
        saved_cfg[
            "refinement"
        ]["method"]
        == "bsrr"
    )

    assert int(
        saved_cfg[
            "refinement"
        ]["spatial_k"]
    ) == 3


    assert (
        saved_cfg[
            "evaluation"
        ][
            "nmi_average_method"
        ]
        == "max"
    )


    # ========================================================
    # Single-factor loss check
    # ========================================================

    expected_loss = (
        expected_ablation_loss(
            dataset,
            variant,
        )
    )


    for key, expected in (
        expected_loss.items()
    ):

        actual = float(
            saved_cfg[
                "loss"
            ][key]
        )

        assert np.isclose(
            actual,
            expected,
        ), (
            f"{name}: "
            f"{key}={actual}, "
            f"expected {expected}"
        )


    # ========================================================
    # Load labels
    # ========================================================

    gt = np.load(
        run_dir
        / "gt_labels.npy"
    )

    pred = np.load(
        run_dir
        / "pred_labels.npy"
    )

    pred_raw = np.load(
        run_dir
        / "pred_concat_z_kmeans.npy"
    )


    # ========================================================
    # Independent metrics
    # ========================================================

    ari = adjusted_rand_score(
        gt,
        pred,
    )

    nmi = normalized_mutual_info_score(
        gt,
        pred,
        average_method="max",
    )


    raw_ari = adjusted_rand_score(
        gt,
        pred_raw,
    )

    raw_nmi = normalized_mutual_info_score(
        gt,
        pred_raw,
        average_method="max",
    )


    # ========================================================
    # metrics.json
    # ========================================================

    with (
        run_dir
        / "metrics.json"
    ).open(
        "r",
        encoding="utf-8",
    ) as f:

        metrics = json.load(f)


    assert np.isclose(
        ari,
        float(metrics["ARI"]),
        atol=1e-12,
    )

    assert np.isclose(
        nmi,
        float(metrics["NMI"]),
        atol=1e-12,
    )


    # ========================================================
    # Loss history
    # ========================================================

    history = metrics.get(
        "loss_history",
        []
    )


    assert len(history) == 400, (
        f"{name}: "
        f"loss_history length "
        f"{len(history)} != 400"
    )


    assert int(
        history[-1]["epoch"]
    ) == 400


    # ========================================================
    # Official repository audit
    # ========================================================

    proc = subprocess.run(
        [
            sys.executable,
            str(
                PROJECT_ROOT
                / "scripts"
                / "audit_run.py"
            ),
            str(run_dir),
        ],
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
    )


    if proc.returncode != 0:

        print(proc.stdout)
        print(proc.stderr)

        raise RuntimeError(
            f"Official audit failed: "
            f"{name}"
        )


    if "PASS" not in proc.stdout:

        print(proc.stdout)

        raise RuntimeError(
            f"Official audit did not "
            f"report PASS: {name}"
        )


    print(
        "\n" + "-" * 90
    )

    print(
        f"AUDIT PASS | "
        f"{dataset} | "
        f"{variant} | "
        f"seed {seed}"
    )

    print("-" * 90)

    print(
        f"Official ARI = "
        f"{ari:.12f}"
    )

    print(
        f"Official NMI = "
        f"{nmi:.12f}"
    )

    print(
        f"Raw ARI      = "
        f"{raw_ari:.12f}"
    )

    print(
        f"Raw NMI      = "
        f"{raw_nmi:.12f}"
    )

    print(
        f"BSRR ΔARI    = "
        f"{ari - raw_ari:+.12f}"
    )

    print(
        f"BSRR ΔNMI    = "
        f"{nmi - raw_nmi:+.12f}"
    )


    print("\nActual loss:")

    for key in [
        "lambda_rec",
        "lambda_mgcl",
        "lambda_cluster",
        "lambda_spatial",
    ]:

        print(
            f"  {key:15s} = "
            f"{saved_cfg['loss'][key]}"
        )


    print(
        "\nOfficial audit_run.py: PASS"
    )


    return {
        "dataset":
            dataset,

        "variant":
            variant,

        "seed":
            seed,

        "ARI":
            ari,

        "NMI":
            nmi,

        "raw_ARI":
            raw_ari,

        "raw_NMI":
            raw_nmi,

        "delta_ARI_BSRR":
            ari - raw_ari,

        "delta_NMI_BSRR":
            nmi - raw_nmi,

        "run_dir":
            str(run_dir),
    }


def run_ablation_config(
    config_path,
):

    config_path = Path(
        config_path
    )


    assert config_path.exists(), (
        f"Missing config: {config_path}"
    )


    with config_path.open(
        "r",
        encoding="utf-8",
    ) as f:

        cfg = yaml.safe_load(f)


    experiment_name = (
        cfg[
            "experiment"
        ]["name"]
    )


    run_dir = (
        PROJECT_ROOT
        / cfg["output"]["root"]
        / experiment_name
    )


    print(
        "\n" + "=" * 100
    )

    print(
        "RUN:",
        experiment_name,
    )

    print(
        "Config:",
        config_path,
    )

    print(
        "Output:",
        run_dir,
    )

    print(
        "=" * 100
    )


    # ========================================================
    # Completed run -> audit only
    # ========================================================

    if (
        run_dir
        / "metrics.json"
    ).exists():

        print(
            "\nExisting completed run "
            "detected -> training skipped."
        )

        return audit_ablation_output(
            config_path,
            run_dir,
        )


    # ========================================================
    # Partial output -> STOP
    #
    # Do not silently overwrite or resume because this runner
    # intentionally contains no new resume logic.
    # ========================================================

    if run_dir.exists():

        contents = list(
            run_dir.iterdir()
        )

        if contents:

            raise RuntimeError(
                "\nPartial ablation run found:\n"
                f"{run_dir}\n\n"
                "Do not automatically resume or overwrite it.\n"
                "Inspect/delete this single incomplete run "
                "before rerunning."
            )


    # ========================================================
    # Fresh training
    # ========================================================

    cmd = [
        sys.executable,
        str(ABLATION_RUNNER),
        "--config",
        str(config_path),
    ]


    print(
        "\nStarting fresh training..."
    )


    subprocess.run(
        cmd,
        cwd=PROJECT_ROOT,
        check=True,
    )


    # ========================================================
    # Final audit
    # ========================================================

    assert (
        run_dir
        / "metrics.json"
    ).exists(), (
        f"Training finished but "
        f"metrics.json not found:\n"
        f"{run_dir}"
    )


    return audit_ablation_output(
        config_path,
        run_dir,
    )


print("=" * 90)
print("ABLATION EXECUTION CONTROLLER")
print("=" * 90)

print(
    "Runner:",
    ABLATION_RUNNER
)

print(
    "Output root:",
    ABLATION_RESULT_ROOT
)

print(
    "\nBehavior:"
)

print(
    "  complete run -> skip + audit"
)

print(
    "  fresh run    -> train + audit"
)

print(
    "  partial run  -> STOP"
)

print(
    "\nPASS: ablation controller ready."
)

print(
    "NO TRAINING performed in this cell."
)

ABLATION EXECUTION CONTROLLER
Runner: /kaggle/working/SpaMGCL/SpaMGCL/experiments/run_exp_ablation_400.py
Output root: /kaggle/working/SpaMGCL/SpaMGCL/results_ablation_400

Behavior:
  complete run -> skip + audit
  fresh run    -> train + audit
  partial run  -> STOP

PASS: ablation controller ready.
NO TRAINING performed in this cell.


Cell 10：Smoke Test 1 — HLN-A1 w/o Sample, seed 0

In [11]:
# ============================================================
# Cell 10
# Formal smoke test 1
# HLN-A1 | w/o Sample CL | seed 0
# ============================================================

HLNA1_WO_SAMPLE_SEED0_CONFIG = (
    ABLATION_CONFIG_DIR
    / (
        "hlna1_ablation_"
        "wo_sample_400ep_seed0.yaml"
    )
)


assert (
    HLNA1_WO_SAMPLE_SEED0_CONFIG.exists()
)


hlna1_wo_sample_seed0 = (
    run_ablation_config(
        HLNA1_WO_SAMPLE_SEED0_CONFIG
    )
)


print(
    "\n" + "=" * 100
)

print(
    "PASS: HLN-A1 "
    "w/o Sample CL "
    "seed0 smoke test complete."
)

print(
    "=" * 100
)


RUN: hlna1_ablation_wo_sample_400ep_seed0
Config: /kaggle/working/SpaMGCL/SpaMGCL/configs/ablation_400ep/hlna1_ablation_wo_sample_400ep_seed0.yaml
Output: /kaggle/working/SpaMGCL/SpaMGCL/results_ablation_400/hlna1_ablation_wo_sample_400ep_seed0

Starting fresh training...


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: HLN-A1 | spots=3484 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3484, 3484), nnz=13206
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=0.252383 | rec=0.252383 | mgcl=8.159293 | cluster=2.951158 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=4.887e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.950 | gradC=0.000e+00
epoch 002/400 | total=0.242171 | rec=0.242171 | mgcl=8.159149 | cluster=2.950280 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=5.133e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.956 | gradC=0.000e+00
epoch 003/400 | total=0.232364 | rec=0.232364 | mgcl=8.159019 | cluster=2.949491 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=5.364e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.962 | gradC=0.000e+00
epoch 00

Cell 11：Smoke Test 2 — HLN-A1 w/o Cluster, seed 0

In [12]:
# ============================================================
# Cell 11
# Formal smoke test 2
# HLN-A1 | w/o Cluster CL | seed 0
# ============================================================

HLNA1_WO_CLUSTER_SEED0_CONFIG = (
    ABLATION_CONFIG_DIR
    / (
        "hlna1_ablation_"
        "wo_cluster_400ep_seed0.yaml"
    )
)


assert (
    HLNA1_WO_CLUSTER_SEED0_CONFIG.exists()
)


hlna1_wo_cluster_seed0 = (
    run_ablation_config(
        HLNA1_WO_CLUSTER_SEED0_CONFIG
    )
)


print(
    "\n" + "=" * 100
)

print(
    "PASS: HLN-A1 "
    "w/o Cluster CL "
    "seed0 smoke test complete."
)

print(
    "=" * 100
)


RUN: hlna1_ablation_wo_cluster_400ep_seed0
Config: /kaggle/working/SpaMGCL/SpaMGCL/configs/ablation_400ep/hlna1_ablation_wo_cluster_400ep_seed0.yaml
Output: /kaggle/working/SpaMGCL/SpaMGCL/results_ablation_400/hlna1_ablation_wo_cluster_400ep_seed0

Starting fresh training...


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: HLN-A1 | spots=3484 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3484, 3484), nnz=13206
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=8.411675 | rec=0.252383 | mgcl=8.159292 | cluster=2.951158 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=4.887e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.950 | gradC=0.000e+00
epoch 002/400 | total=8.400130 | rec=0.242314 | mgcl=8.157817 | cluster=2.950293 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=2.620e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.956 | gradC=0.000e+00
epoch 003/400 | total=8.389496 | rec=0.232629 | mgcl=8.156867 | cluster=2.949508 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=4.570e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.962 | gradC=0.000e+00
epoch 00

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (10). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)



epoch 372/400 | total=6.165539 | rec=0.014014 | mgcl=6.151525 | cluster=2.944439 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=5.568e-04 | neg_count=12134772 | snf_masked_positions=0 | effC=10.000 | gradC=0.000e+00
epoch 373/400 | total=6.168024 | rec=0.013988 | mgcl=6.154035 | cluster=2.944439 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=7.496e-04 | neg_count=12134772 | snf_masked_positions=0 | effC=10.000 | gradC=0.000e+00
epoch 374/400 | total=6.167775 | rec=0.013961 | mgcl=6.153814 | cluster=2.944439 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=5.748e-04 | neg_count=12134772 | snf_masked_positions=0 | effC=10.000 | gradC=0.000e+00
epoch 375/400 | total=6.165791 | rec=0.013936 | mgcl=6.151855 | cluster=2.944439 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=6.680e-04 |

Cell 12：完成 HLN-A1 剩余 8 个训练 run

In [13]:
# ============================================================
# Cell 12
# HLN-A1 remaining formal ablation runs
#
# wo_sample : seeds 1-4
# wo_cluster: seeds 1-4
#
# 8 new runs
# ============================================================

import time


HLNA1_REMAINING_RESULTS = []


jobs = []

for variant in [
    "wo_sample",
    "wo_cluster",
]:

    for seed in [
        1,
        2,
        3,
        4,
    ]:

        config_path = (
            ABLATION_CONFIG_DIR
            / (
                f"hlna1_ablation_"
                f"{variant}_"
                f"400ep_seed{seed}.yaml"
            )
        )

        jobs.append(
            (
                variant,
                seed,
                config_path,
            )
        )


print("=" * 100)
print("HLN-A1 FORMAL ABLATION")
print("Remaining 8 runs")
print("=" * 100)


for i, (
    variant,
    seed,
    config_path,
) in enumerate(
    jobs,
    start=1,
):

    print(
        "\n" + "#" * 100
    )

    print(
        f"{i}/8 | "
        f"HLN-A1 | "
        f"{variant} | "
        f"seed {seed}"
    )

    print(
        "#" * 100
    )


    t0 = time.time()


    result = run_ablation_config(
        config_path
    )


    HLNA1_REMAINING_RESULTS.append(
        result
    )


    elapsed = (
        time.time() - t0
    ) / 60.0


    print(
        f"\nFinished in "
        f"{elapsed:.2f} min"
    )


print(
    "\n" + "=" * 100
)

print(
    "PASS: HLN-A1 remaining "
    "8 ablation runs complete."
)

print(
    "=" * 100
)

HLN-A1 FORMAL ABLATION
Remaining 8 runs

####################################################################################################
1/8 | HLN-A1 | wo_sample | seed 1
####################################################################################################

RUN: hlna1_ablation_wo_sample_400ep_seed1
Config: /kaggle/working/SpaMGCL/SpaMGCL/configs/ablation_400ep/hlna1_ablation_wo_sample_400ep_seed1.yaml
Output: /kaggle/working/SpaMGCL/SpaMGCL/results_ablation_400/hlna1_ablation_wo_sample_400ep_seed1

Starting fresh training...


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: HLN-A1 | spots=3484 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3484, 3484), nnz=13206
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=0.237463 | rec=0.237463 | mgcl=8.159257 | cluster=2.950006 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=2.644e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.959 | gradC=0.000e+00
epoch 002/400 | total=0.228589 | rec=0.228589 | mgcl=8.159019 | cluster=2.949121 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=2.659e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.966 | gradC=0.000e+00
epoch 003/400 | total=0.220121 | rec=0.220121 | mgcl=8.158794 | cluster=2.948352 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=2.680e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.972 | gradC=0.000e+00
epoch 00

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: HLN-A1 | spots=3484 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3484, 3484), nnz=13206
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=0.254117 | rec=0.254117 | mgcl=8.157986 | cluster=2.951155 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=9.127e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.950 | gradC=0.000e+00
epoch 002/400 | total=0.244267 | rec=0.244267 | mgcl=8.157870 | cluster=2.950444 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=9.100e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.955 | gradC=0.000e+00
epoch 003/400 | total=0.234902 | rec=0.234902 | mgcl=8.157760 | cluster=2.949769 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=9.071e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.959 | gradC=0.000e+00
epoch 00

...


epoch 372/400 | total=6.159428 | rec=0.016238 | mgcl=6.143189 | cluster=2.944439 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=6.182e-04 | neg_count=12134772 | snf_masked_positions=0 | effC=10.000 | gradC=0.000e+00
epoch 373/400 | total=6.157482 | rec=0.016198 | mgcl=6.141283 | cluster=2.944439 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=3.883e-04 | neg_count=12134772 | snf_masked_positions=0 | effC=10.000 | gradC=0.000e+00
epoch 374/400 | total=6.156569 | rec=0.016158 | mgcl=6.140412 | cluster=2.944439 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=3.395e-04 | neg_count=12134772 | snf_masked_positions=0 | effC=10.000 | gradC=0.000e+00
epoch 375/400 | total=6.156422 | rec=0.016122 | mgcl=6.140300 | cluster=2.944439 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=4.889e-04 |

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Dataset: HLN-A1 | spots=3484 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3484, 3484), nnz=13206
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=8.424172 | rec=0.265092 | mgcl=8.159081 | cluster=2.950062 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=5.494e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.953 | gradC=0.000e+00
epoch 002/400 | total=8.412754 | rec=0.255082 | mgcl=8.157672 | cluster=2.949080 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=6.366e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.962 | gradC=0.000e+00
epoch 003/400 | total=8.402320 | rec=0.245542 | mgcl=8.156777 | cluster=2.948241 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=9.382e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.969 | gradC=0.000e+00
epoch 00

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (10). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)



epoch 372/400 | total=6.199154 | rec=0.014737 | mgcl=6.184417 | cluster=2.944439 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=3.297e-04 | neg_count=12134772 | snf_masked_positions=0 | effC=10.000 | gradC=0.000e+00
epoch 373/400 | total=6.196173 | rec=0.014716 | mgcl=6.181457 | cluster=2.944439 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=4.214e-04 | neg_count=12134772 | snf_masked_positions=0 | effC=10.000 | gradC=0.000e+00
epoch 374/400 | total=6.196126 | rec=0.014690 | mgcl=6.181436 | cluster=2.944439 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=5.829e-04 | neg_count=12134772 | snf_masked_positions=0 | effC=10.000 | gradC=0.000e+00
epoch 375/400 | total=6.197579 | rec=0.014662 | mgcl=6.182917 | cluster=2.944439 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=3.530e-04 |

Cell 12：完成 HLN-A1 剩余 8 个正式消融 run

In [15]:
# ============================================================
# Cell 12
# Complete remaining HLN-A1 formal ablation runs
#
# Already completed:
#   wo_sample  seed0
#   wo_cluster seed0
#
# This cell runs:
#   wo_sample  seeds 1-4
#   wo_cluster seeds 1-4
#
# Total new runs here = 8
# ============================================================

from pathlib import Path
import time


print("=" * 100)
print("HLN-A1 FORMAL ABLATION")
print("Remaining seeds 1-4")
print("=" * 100)


jobs = []


for variant in [
    "wo_sample",
    "wo_cluster",
]:

    for seed in [
        1,
        2,
        3,
        4,
    ]:

        config_path = (
            ABLATION_CONFIG_DIR
            / (
                f"hlna1_ablation_"
                f"{variant}_"
                f"400ep_seed{seed}.yaml"
            )
        )

        assert config_path.exists(), (
            f"Missing config:\n{config_path}"
        )

        jobs.append(
            {
                "variant": variant,
                "seed": seed,
                "config": config_path,
            }
        )


assert len(jobs) == 8


HLNA1_REMAINING_RESULTS = []


for i, job in enumerate(
    jobs,
    start=1,
):

    print(
        "\n" + "#" * 100
    )

    print(
        f"{i}/8 | "
        f"HLN-A1 | "
        f"{job['variant']} | "
        f"seed {job['seed']}"
    )

    print(
        "#" * 100
    )


    t0 = time.time()


    result = run_ablation_config(
        job["config"]
    )


    elapsed_min = (
        time.time()
        - t0
    ) / 60.0


    HLNA1_REMAINING_RESULTS.append(
        result
    )


    print(
        f"\nFinished run "
        f"{i}/8 in "
        f"{elapsed_min:.2f} min"
    )


print(
    "\n" + "=" * 100
)

print(
    "PASS: remaining 8 HLN-A1 "
    "formal ablation runs complete."
)

print(
    "=" * 100
)


# ============================================================
# Final completeness check:
# 2 variants × 5 seeds = 10 completed runs
# ============================================================

expected_runs = []


for variant in [
    "wo_sample",
    "wo_cluster",
]:

    for seed in range(5):

        run_dir = (
            ABLATION_RESULT_ROOT
            / (
                f"hlna1_ablation_"
                f"{variant}_"
                f"400ep_seed{seed}"
            )
        )

        expected_runs.append(
            run_dir
        )


for run_dir in expected_runs:

    assert run_dir.exists(), (
        f"Missing run directory:\n"
        f"{run_dir}"
    )

    assert (
        run_dir
        / "metrics.json"
    ).exists(), (
        f"Missing metrics.json:\n"
        f"{run_dir}"
    )

    assert (
        run_dir
        / "pred_labels.npy"
    ).exists(), (
        f"Missing pred_labels.npy:\n"
        f"{run_dir}"
    )

    assert (
        run_dir
        / "config.yaml"
    ).exists(), (
        f"Missing config.yaml:\n"
        f"{run_dir}"
    )


print(
    "\nPASS: HLN-A1 has "
    "10/10 completed ablation runs."
)

print(
    "  wo_sample  : 5/5"
)

print(
    "  wo_cluster : 5/5"
)

HLN-A1 FORMAL ABLATION
Remaining seeds 1-4

####################################################################################################
1/8 | HLN-A1 | wo_sample | seed 1
####################################################################################################

RUN: hlna1_ablation_wo_sample_400ep_seed1
Config: /kaggle/working/SpaMGCL/SpaMGCL/configs/ablation_400ep/hlna1_ablation_wo_sample_400ep_seed1.yaml
Output: /kaggle/working/SpaMGCL/SpaMGCL/results_ablation_400/hlna1_ablation_wo_sample_400ep_seed1

Existing completed run detected -> training skipped.

------------------------------------------------------------------------------------------
AUDIT PASS | HLN-A1 | wo_sample | seed 1
------------------------------------------------------------------------------------------
Official ARI = 0.167628686371
Official NMI = 0.276527120044
Raw ARI      = 0.166145294316
Raw NMI      = 0.275209108885
BSRR ΔARI    = +0.001483392054
BSRR ΔNMI    = +0.001318011159

Actual loss:


Cell 13：收集当前 10 个消融结果并加入冻结 Full

In [17]:
# ============================================================
# Cell 13
# Collect HLN-A1 current ablation results
#
# Local runs:
#   w/o Sample CL  seeds 0-4
#   w/o Cluster CL seeds 0-4
#
# Reference:
#   frozen formal Full seeds 0-4
#
# NOTE:
# Full values are copied from the previously verified
# formal 400-epoch experiment. No retraining here.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)

SUMMARY_DIR = (
    PROJECT_ROOT
    / "ablation_summary_400"
)

SUMMARY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# Frozen formal Full results
# seeds 0-4
# ============================================================

FULL_REFERENCE = {
    0: {
        "ARI": 0.220614,
        "NMI": 0.358325,
    },
    1: {
        "ARI": 0.273804,
        "NMI": 0.393244,
    },
    2: {
        "ARI": 0.278258,
        "NMI": 0.404178,
    },
    3: {
        "ARI": 0.237522,
        "NMI": 0.353609,
    },
    4: {
        "ARI": 0.263325,
        "NMI": 0.387647,
    },
}


rows = []


# ============================================================
# Full reference
# ============================================================

for seed in range(5):

    rows.append(
        {
            "dataset": "HLN-A1",
            "variant": "Full",
            "seed": seed,
            "ARI": FULL_REFERENCE[seed]["ARI"],
            "NMI": FULL_REFERENCE[seed]["NMI"],
            "source": "frozen_formal_reference",
        }
    )


# ============================================================
# Local ablation runs
# ============================================================

variant_map = {
    "wo_sample": "w/o Sample CL",
    "wo_cluster": "w/o Cluster CL",
}


for variant_key, variant_label in variant_map.items():

    for seed in range(5):

        run_dir = (
            ABLATION_RESULT_ROOT
            / (
                f"hlna1_ablation_"
                f"{variant_key}_"
                f"400ep_seed{seed}"
            )
        )

        assert run_dir.exists(), (
            f"Missing run:\n{run_dir}"
        )

        gt = np.load(
            run_dir / "gt_labels.npy"
        )

        pred = np.load(
            run_dir / "pred_labels.npy"
        )

        ari = adjusted_rand_score(
            gt,
            pred,
        )

        nmi = normalized_mutual_info_score(
            gt,
            pred,
            average_method="max",
        )

        rows.append(
            {
                "dataset": "HLN-A1",
                "variant": variant_label,
                "seed": seed,
                "ARI": ari,
                "NMI": nmi,
                "source": "local_ablation_run",
            }
        )


ablation_raw = pd.DataFrame(
    rows
)


VARIANT_ORDER = [
    "Full",
    "w/o Sample CL",
    "w/o Cluster CL",
]


ablation_raw["variant"] = (
    pd.Categorical(
        ablation_raw["variant"],
        categories=VARIANT_ORDER,
        ordered=True,
    )
)


ablation_raw = (
    ablation_raw
    .sort_values(
        [
            "variant",
            "seed",
        ]
    )
    .reset_index(
        drop=True
    )
)


print("=" * 100)
print("HLN-A1 CURRENT ABLATION RAW RESULTS")
print("=" * 100)

print(
    ablation_raw.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)


raw_path = (
    SUMMARY_DIR
    / "HLNA1_ablation_current_5seeds_raw.csv"
)

ablation_raw.to_csv(
    raw_path,
    index=False,
)


print("\nSaved:")
print(raw_path)

HLN-A1 CURRENT ABLATION RAW RESULTS
dataset        variant  seed      ARI      NMI                  source
 HLN-A1           Full     0 0.220614 0.358325 frozen_formal_reference
 HLN-A1           Full     1 0.273804 0.393244 frozen_formal_reference
 HLN-A1           Full     2 0.278258 0.404178 frozen_formal_reference
 HLN-A1           Full     3 0.237522 0.353609 frozen_formal_reference
 HLN-A1           Full     4 0.263325 0.387647 frozen_formal_reference
 HLN-A1  w/o Sample CL     0 0.223569 0.333768      local_ablation_run
 HLN-A1  w/o Sample CL     1 0.167629 0.276527      local_ablation_run
 HLN-A1  w/o Sample CL     2 0.199565 0.321016      local_ablation_run
 HLN-A1  w/o Sample CL     3 0.208491 0.320064      local_ablation_run
 HLN-A1  w/o Sample CL     4 0.175976 0.290227      local_ablation_run
 HLN-A1 w/o Cluster CL     0 0.264554 0.392234      local_ablation_run
 HLN-A1 w/o Cluster CL     1 0.282398 0.383897      local_ablation_run
 HLN-A1 w/o Cluster CL     2 0.275098 0.4

Cell 15：做 paired difference 并保存

In [18]:
# ============================================================
# Cell 15
# Paired comparison against Full
#
# Delta = Ablation - Full
#
# Delta < 0 : removing module hurts performance
# Delta > 0 : removing module improves performance
# ============================================================

full = (
    ablation_raw[
        ablation_raw["variant"]
        == "Full"
    ]
    .set_index("seed")
    .sort_index()
)


paired_rows = []


for variant in [
    "w/o Sample CL",
    "w/o Cluster CL",
]:

    d = (
        ablation_raw[
            ablation_raw["variant"]
            == variant
        ]
        .set_index("seed")
        .sort_index()
    )

    for seed in range(5):

        paired_rows.append(
            {
                "Variant": variant,
                "Seed": seed,

                "Full_ARI":
                    full.loc[
                        seed,
                        "ARI"
                    ],

                "Ablation_ARI":
                    d.loc[
                        seed,
                        "ARI"
                    ],

                "Delta_ARI":
                    (
                        d.loc[
                            seed,
                            "ARI"
                        ]
                        -
                        full.loc[
                            seed,
                            "ARI"
                        ]
                    ),

                "Full_NMI":
                    full.loc[
                        seed,
                        "NMI"
                    ],

                "Ablation_NMI":
                    d.loc[
                        seed,
                        "NMI"
                    ],

                "Delta_NMI":
                    (
                        d.loc[
                            seed,
                            "NMI"
                        ]
                        -
                        full.loc[
                            seed,
                            "NMI"
                        ]
                    ),
            }
        )


paired_df = pd.DataFrame(
    paired_rows
)


print("=" * 105)
print("HLN-A1 PAIRED DIFFERENCE")
print("Delta = Ablation - Full")
print("=" * 105)

print(
    paired_df.to_string(
        index=False,
        float_format=lambda x: f"{x:+.6f}",
    )
)


paired_summary_rows = []


for variant in [
    "w/o Sample CL",
    "w/o Cluster CL",
]:

    d = paired_df[
        paired_df["Variant"]
        == variant
    ]

    da = d["Delta_ARI"].to_numpy()
    dn = d["Delta_NMI"].to_numpy()

    paired_summary_rows.append(
        {
            "Variant": variant,

            "Mean_Delta_ARI":
                np.mean(da),

            "Mean_Delta_NMI":
                np.mean(dn),

            "Full_better_ARI":
                int(
                    np.sum(
                        da < 0
                    )
                ),

            "Full_better_NMI":
                int(
                    np.sum(
                        dn < 0
                    )
                ),
        }
    )


paired_summary = pd.DataFrame(
    paired_summary_rows
)


print(
    "\n" + "=" * 105
)

print(
    "PAIRED SUMMARY"
)

print(
    "=" * 105
)

print(
    paired_summary.to_string(
        index=False,
        float_format=lambda x: f"{x:+.6f}",
    )
)


paired_path = (
    SUMMARY_DIR
    / "HLNA1_ablation_current_5seeds_paired.csv"
)

paired_summary_path = (
    SUMMARY_DIR
    / "HLNA1_ablation_current_5seeds_paired_summary.csv"
)


paired_df.to_csv(
    paired_path,
    index=False,
)

paired_summary.to_csv(
    paired_summary_path,
    index=False,
)


print("\nSaved:")
print(paired_path)
print(paired_summary_path)

HLN-A1 PAIRED DIFFERENCE
Delta = Ablation - Full
       Variant  Seed  Full_ARI  Ablation_ARI  Delta_ARI  Full_NMI  Ablation_NMI  Delta_NMI
 w/o Sample CL     0 +0.220614     +0.223569  +0.002955 +0.358325     +0.333768  -0.024557
 w/o Sample CL     1 +0.273804     +0.167629  -0.106175 +0.393244     +0.276527  -0.116717
 w/o Sample CL     2 +0.278258     +0.199565  -0.078693 +0.404178     +0.321016  -0.083162
 w/o Sample CL     3 +0.237522     +0.208491  -0.029031 +0.353609     +0.320064  -0.033545
 w/o Sample CL     4 +0.263325     +0.175976  -0.087349 +0.387647     +0.290227  -0.097420
w/o Cluster CL     0 +0.220614     +0.264554  +0.043940 +0.358325     +0.392234  +0.033909
w/o Cluster CL     1 +0.273804     +0.282398  +0.008594 +0.393244     +0.383897  -0.009347
w/o Cluster CL     2 +0.278258     +0.275098  -0.003160 +0.404178     +0.403331  -0.000847
w/o Cluster CL     3 +0.237522     +0.213475  -0.024047 +0.353609     +0.330049  -0.023560
w/o Cluster CL     4 +0.263325     +0.261

Cell 16：冻结当前 HLN-A1 消融结果

In [19]:
# ============================================================
# Cell 16
# Freeze current HLN-A1 ablation experiment
#
# Save:
#   - raw / summary / paired CSV
#   - configs
#   - metrics.json
#   - manifest.json if available
#   - gt_labels.npy
#   - pred_labels.npy
#   - pred_concat_z_kmeans.npy
#
# No checkpoints / embeddings -> lightweight archive
# ============================================================

from pathlib import Path
import shutil
import hashlib
import json
import numpy as np
import pandas as pd


FREEZE_ROOT = (
    PROJECT_ROOT
    / "HLNA1_ablation_current_5seeds_FREEZE"
)

if FREEZE_ROOT.exists():
    shutil.rmtree(FREEZE_ROOT)

FREEZE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 1. Rebuild / save final summary table
# ============================================================

summary_rows = []

for variant in [
    "Full",
    "w/o Sample CL",
    "w/o Cluster CL",
]:

    d = (
        ablation_raw[
            ablation_raw["variant"]
            == variant
        ]
        .sort_values("seed")
    )

    ari = d["ARI"].to_numpy()
    nmi = d["NMI"].to_numpy()

    summary_rows.append(
        {
            "Dataset": "HLN-A1",
            "Variant": variant,

            "ARI_mean":
                np.mean(ari),

            "ARI_std":
                np.std(
                    ari,
                    ddof=0,
                ),

            "NMI_mean":
                np.mean(nmi),

            "NMI_std":
                np.std(
                    nmi,
                    ddof=0,
                ),

            "ARI_mean_std":
                (
                    f"{np.mean(ari):.6f} "
                    f"± "
                    f"{np.std(ari, ddof=0):.6f}"
                ),

            "NMI_mean_std":
                (
                    f"{np.mean(nmi):.6f} "
                    f"± "
                    f"{np.std(nmi, ddof=0):.6f}"
                ),
        }
    )


final_summary = pd.DataFrame(
    summary_rows
)


final_summary_path = (
    FREEZE_ROOT
    / "HLNA1_ablation_current_5seeds_TABLE.csv"
)

final_summary.to_csv(
    final_summary_path,
    index=False,
)


# raw
ablation_raw.to_csv(
    FREEZE_ROOT
    / "HLNA1_ablation_current_5seeds_raw.csv",
    index=False,
)


# paired
paired_df.to_csv(
    FREEZE_ROOT
    / "HLNA1_ablation_current_5seeds_paired.csv",
    index=False,
)


paired_summary.to_csv(
    FREEZE_ROOT
    / "HLNA1_ablation_current_5seeds_paired_summary.csv",
    index=False,
)


# ============================================================
# 2. Save frozen Full reference explicitly
# ============================================================

FULL_REFERENCE_TO_SAVE = {
    str(seed): {
        "ARI":
            float(
                FULL_REFERENCE[seed]["ARI"]
            ),

        "NMI":
            float(
                FULL_REFERENCE[seed]["NMI"]
            ),
    }

    for seed in range(5)
}


protocol = {
    "dataset":
        "HLN-A1",

    "epochs":
        400,

    "seeds":
        [0, 1, 2, 3, 4],

    "warm_up_epochs":
        10,

    "official_readout":
        "concat_z -> BSRR -> KMeans",

    "bsrr_spatial_k":
        3,

    "kmeans_n_init":
        20,

    "kmeans_random_state":
        0,

    "nmi_average_method":
        "max",

    "std_ddof":
        0,

    "variants":
        [
            "Full",
            "w/o Sample CL",
            "w/o Cluster CL",
        ],

    "full_source":
        (
            "Previously frozen and verified "
            "400-epoch formal experiment"
        ),

    "full_reference":
        FULL_REFERENCE_TO_SAVE,
}


with (
    FREEZE_ROOT
    / "protocol.json"
).open(
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        protocol,
        f,
        indent=2,
    )


# ============================================================
# 3. Copy 10 ablation configs
# ============================================================

config_out = (
    FREEZE_ROOT
    / "configs"
)

config_out.mkdir(
    parents=True,
    exist_ok=True,
)


for variant in [
    "wo_sample",
    "wo_cluster",
]:

    for seed in range(5):

        src = (
            ABLATION_CONFIG_DIR
            / (
                f"hlna1_ablation_"
                f"{variant}_"
                f"400ep_seed{seed}.yaml"
            )
        )

        assert src.exists(), (
            f"Missing config:\n{src}"
        )

        shutil.copy2(
            src,
            config_out / src.name,
        )


# ============================================================
# 4. Copy essential run evidence
# ============================================================

runs_out = (
    FREEZE_ROOT
    / "runs"
)

runs_out.mkdir(
    parents=True,
    exist_ok=True,
)


ESSENTIAL_FILES = [
    "metrics.json",
    "config.yaml",
    "manifest.json",
    "gt_labels.npy",
    "pred_labels.npy",
    "pred_concat_z_kmeans.npy",
]


for variant in [
    "wo_sample",
    "wo_cluster",
]:

    for seed in range(5):

        run_name = (
            f"hlna1_ablation_"
            f"{variant}_"
            f"400ep_seed{seed}"
        )

        src_dir = (
            ABLATION_RESULT_ROOT
            / run_name
        )

        assert src_dir.exists()

        dst_dir = (
            runs_out
            / run_name
        )

        dst_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        for filename in ESSENTIAL_FILES:

            src = (
                src_dir
                / filename
            )

            if src.exists():

                shutil.copy2(
                    src,
                    dst_dir / filename,
                )


# ============================================================
# 5. SHA256 manifest
# ============================================================

hash_rows = []


for path in sorted(
    FREEZE_ROOT.rglob("*")
):

    if not path.is_file():
        continue

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):

            h.update(block)

    hash_rows.append(
        {
            "file":
                str(
                    path.relative_to(
                        FREEZE_ROOT
                    )
                ),

            "sha256":
                h.hexdigest(),
        }
    )


hash_df = pd.DataFrame(
    hash_rows
)

hash_df.to_csv(
    FREEZE_ROOT
    / "SHA256_MANIFEST.csv",
    index=False,
)


# ============================================================
# 6. Create ZIP
# ============================================================

archive_base = str(
    PROJECT_ROOT
    / "HLNA1_ablation_current_5seeds_FREEZE"
)


zip_path = shutil.make_archive(
    archive_base,
    "zip",
    root_dir=FREEZE_ROOT,
)


print("=" * 100)
print("HLN-A1 ABLATION FREEZE COMPLETE")
print("=" * 100)

print(
    final_summary[
        [
            "Variant",
            "ARI_mean_std",
            "NMI_mean_std",
        ]
    ].to_string(
        index=False
    )
)

print("\nFrozen directory:")
print(FREEZE_ROOT)

print("\nZIP:")
print(zip_path)

print(
    "\nPASS: current HLN-A1 "
    "ablation results frozen."
)

HLN-A1 ABLATION FREEZE COMPLETE
       Variant        ARI_mean_std        NMI_mean_std
          Full 0.254705 ± 0.022142 0.379401 ± 0.019915
 w/o Sample CL 0.195046 ± 0.020640 0.308321 ± 0.021377
w/o Cluster CL 0.259372 ± 0.024145 0.380506 ± 0.025971

Frozen directory:
/kaggle/working/SpaMGCL/SpaMGCL/HLNA1_ablation_current_5seeds_FREEZE

ZIP:
/kaggle/working/SpaMGCL/SpaMGCL/HLNA1_ablation_current_5seeds_FREEZE.zip

PASS: current HLN-A1 ablation results frozen.


Cell 17：检查源码里的 Fine / Coarse 实现

In [20]:
# ============================================================
# Cell 17
# Source inspection for multigranularity implementation
#
# READ ONLY
# DO NOT MODIFY SOURCE
# ============================================================

from pathlib import Path
import re


SEARCH_ROOTS = [
    PROJECT_ROOT / "src",
    PROJECT_ROOT / "experiments",
]


KEYWORDS = [
    "fine",
    "coarse",
    "granular",
    "granularity",
    "multi_gran",
    "multigran",
    "mgcl",
    "mgcl_weight",
]


matches = []


for root in SEARCH_ROOTS:

    if not root.exists():
        continue

    for path in root.rglob("*.py"):

        try:
            lines = path.read_text(
                encoding="utf-8"
            ).splitlines()

        except UnicodeDecodeError:

            lines = path.read_text(
                encoding="utf-8",
                errors="ignore",
            ).splitlines()


        for lineno, line in enumerate(
            lines,
            start=1,
        ):

            lower = line.lower()

            if any(
                keyword in lower
                for keyword in KEYWORDS
            ):

                matches.append(
                    (
                        path,
                        lineno,
                        line,
                    )
                )


print("=" * 110)
print("SpaMGCL MULTIGRANULARITY SOURCE SEARCH")
print("=" * 110)

print(
    "Total matching lines:",
    len(matches),
)


current_file = None


for path, lineno, line in matches:

    if path != current_file:

        current_file = path

        print(
            "\n" + "=" * 110
        )

        print(
            path.relative_to(
                PROJECT_ROOT
            )
        )

        print(
            "=" * 110
        )

    print(
        f"{lineno:4d} | {line}"
    )


print(
    "\n" + "=" * 110
)

print(
    "PASS: source inspection complete."
)

print(
    "NO source files were modified."
)

SpaMGCL MULTIGRANULARITY SOURCE SEARCH
Total matching lines: 234

src/__init__.py
   1 | """SpaMGCL implementation package."""

src/models/multigranularity.py
   1 | """MGCMVC-style fine/coarse/multigranularity representations.
   5 | refinement logic, so it can serve as the MGCMVC-original migration.
  17 | class FineGrainedEncoder(nn.Module):
  65 | class CoarseGrainedMLP(nn.Module):
  66 |     """View-specific MLP mapping ``Z_v`` to coarse representation ``H_v``."""
  71 |         coarse_dim: int,
  75 |         hidden_dim = hidden_dim or max(coarse_dim, min(128, input_dim))
  79 |             nn.Linear(hidden_dim, coarse_dim),
  88 | class MultigranularityFusion(nn.Module):
  93 |         fine_dim: int,
  94 |         coarse_dim: int,
 100 |         hidden_dim = hidden_dim or max(output_dim, min(256, fine_dim + coarse_dim))
 102 |             nn.Linear(fine_dim + coarse_dim, hidden_dim),
 117 | class ViewMultiGranularityEncoder(nn.Module):
 123 |         fine_dim: int,
 124 |      

Cell 18：把真正相关的源码上下文完整打印出来

In [21]:
# ============================================================
# Cell 18
# Exact source inspection for Fine / Coarse ablation design
#
# READ ONLY
# DO NOT MODIFY SOURCE
# ============================================================

from pathlib import Path


FILES_TO_INSPECT = {
    "multigranularity": (
        PROJECT_ROOT
        / "src/models/multigranularity.py"
    ),

    "spamgcl": (
        PROJECT_ROOT
        / "src/models/spamgcl.py"
    ),

    "runner": (
        PROJECT_ROOT
        / "experiments/run_exp.py"
    ),
}


def show_lines(
    path,
    start,
    end,
):
    """
    Print exact numbered source lines,
    inclusive [start, end].
    """

    assert path.exists(), (
        f"Missing source file:\n{path}"
    )

    lines = path.read_text(
        encoding="utf-8"
    ).splitlines()

    print(
        "\n" + "=" * 120
    )

    print(
        f"{path.relative_to(PROJECT_ROOT)}"
        f" | lines {start}-{end}"
    )

    print(
        "=" * 120
    )

    for lineno in range(
        start,
        min(
            end,
            len(lines),
        ) + 1,
    ):

        print(
            f"{lineno:4d} | "
            f"{lines[lineno - 1]}"
        )


# ============================================================
# 1. Fine / Coarse / Fusion exact implementation
# ============================================================

show_lines(
    FILES_TO_INSPECT[
        "multigranularity"
    ],
    1,
    150,
)


# ============================================================
# 2. SpaMGCL forward:
#    z / h / g
#    sample contrastive
#    cluster loss
#    total loss
# ============================================================

show_lines(
    FILES_TO_INSPECT[
        "spamgcl"
    ],
    100,
    230,
)


# ============================================================
# 3. Runner embedding construction
#    concat_g / concat_z / z_mean etc.
# ============================================================

show_lines(
    FILES_TO_INSPECT[
        "runner"
    ],
    405,
    445,
)


# ============================================================
# 4. Official readout
# ============================================================

show_lines(
    FILES_TO_INSPECT[
        "runner"
    ],
    450,
    510,
)


# ============================================================
# 5. Model construction / dimensions
# ============================================================

show_lines(
    FILES_TO_INSPECT[
        "runner"
    ],
    490,
    530,
)


show_lines(
    FILES_TO_INSPECT[
        "runner"
    ],
    700,
    735,
)


# ============================================================
# 6. Training loss reconstruction
# ============================================================

show_lines(
    FILES_TO_INSPECT[
        "runner"
    ],
    800,
    835,
)


# ============================================================
# 7. Final saved embeddings
# ============================================================

show_lines(
    FILES_TO_INSPECT[
        "runner"
    ],
    980,
    1070,
)


print(
    "\n" + "=" * 120
)

print(
    "PASS: exact Fine / Coarse "
    "source context printed."
)

print(
    "NO source files were modified."
)


src/models/multigranularity.py | lines 1-150
   1 | """MGCMVC-style fine/coarse/multigranularity representations.
   2 | 
   3 | This module contains the representation path only. It has no spatial
   4 | coordinates, spatial consistency term, spatial negative mask, or cluster
   5 | refinement logic, so it can serve as the MGCMVC-original migration.
   6 | """
   7 | 
   8 | from __future__ import annotations
   9 | 
  10 | from typing import Dict, Iterable, List, Mapping, Optional, Sequence, Tuple
  11 | 
  12 | import torch
  13 | from torch import Tensor, nn
  14 | import torch.nn.functional as F
  15 | 
  16 | 
  17 | class FineGrainedEncoder(nn.Module):
  18 |     """Shallow view-specific autoencoder path producing ``Z_v``.
  19 | 
  20 |     ``X_v`` has shape ``(N, input_dim)``. The encoder returns ``Z_v`` with
  21 |     shape ``(N, latent_dim)`` and the decoder reconstructs ``X_v``.
  22 |     """
  23 | 
  24 |     def __init__(
  25 |         self,
  26 |         input_dim:

Cell 19：创建专用 multigranularity ablation runner

In [27]:
# ============================================================
# Cell 19
# Multigranularity ablation runner - CLEAN V2
#
# Variants:
#   fine_only
#   coarse_only
#   linear_fusion
#
# IMPORTANT:
#   - Core src/ is NOT modified
#   - Output dimension always equals representation_dim
#   - Official readout remains concat_z -> BSRR -> KMeans
# ============================================================

from pathlib import Path
import py_compile


BASE_RUNNER = (
    PROJECT_ROOT
    / "experiments"
    / "run_exp_ablation_400.py"
)

MG_RUNNER = (
    PROJECT_ROOT
    / "experiments"
    / "run_exp_mg_ablation_400.py"
)


assert BASE_RUNNER.exists(), (
    f"Missing base runner:\n{BASE_RUNNER}"
)


text = BASE_RUNNER.read_text(
    encoding="utf-8"
)


# ============================================================
# 1. Add required imports
# ============================================================

import_marker = "import torch\n"

assert import_marker in text


text = text.replace(
    import_marker,
    (
        "import torch\n"
        "from torch import nn\n"
        "import torch.nn.functional as F\n"
    ),
    1,
)


# ============================================================
# 2. Add dimension-compatible MG ablation modules
# ============================================================

insert_marker = "def _build_graphs("

assert insert_marker in text


helper_code = r'''
class _FineOnlyFusion(nn.Module):
    """
    Fine-only representation:
        z -> Linear -> normalize -> g

    Output dimension equals representation_dim.
    """

    def __init__(
        self,
        fine_dim: int,
        output_dim: int,
    ) -> None:
        super().__init__()

        self.projection = nn.Linear(
            fine_dim,
            output_dim,
        )

    def forward(
        self,
        z,
        h,
    ):
        return F.normalize(
            self.projection(z),
            dim=1,
        )


class _CoarseOnlyFusion(nn.Module):
    """
    Coarse-only representation:
        h -> Linear -> normalize -> g

    Output dimension equals representation_dim.
    """

    def __init__(
        self,
        coarse_dim: int,
        output_dim: int,
    ) -> None:
        super().__init__()

        self.projection = nn.Linear(
            coarse_dim,
            output_dim,
        )

    def forward(
        self,
        z,
        h,
    ):
        return F.normalize(
            self.projection(h),
            dim=1,
        )


class _LinearFusion(nn.Module):
    """
    Replace Full nonlinear fusion:

        concat(z,h)
        -> Linear
        -> ReLU
        -> Linear
        -> normalize

    with:

        concat(z,h)
        -> single Linear
        -> normalize
    """

    def __init__(
        self,
        fine_dim: int,
        coarse_dim: int,
        output_dim: int,
    ) -> None:
        super().__init__()

        self.projection = nn.Linear(
            fine_dim + coarse_dim,
            output_dim,
        )

    def forward(
        self,
        z,
        h,
    ):
        x = torch.cat(
            [z, h],
            dim=1,
        )

        return F.normalize(
            self.projection(x),
            dim=1,
        )


def _apply_multigranularity_ablation(
    model,
    config,
):
    """
    Replace only the per-view fusion module.

    Modes
    -----
    full:
        Original learned nonlinear Fusion(z,h)

    fine_only:
        Fine representation z only

    coarse_only:
        Coarse representation h only

    linear_fusion:
        z+h retained, but nonlinear fusion MLP replaced
        by one linear projection.
    """

    section = config.get(
        "multigranularity_ablation",
        {}
    )

    mode = str(
        section.get(
            "mode",
            "full",
        )
    ).lower()


    valid_modes = {
        "full",
        "fine_only",
        "coarse_only",
        "linear_fusion",
    }


    if mode not in valid_modes:
        raise ValueError(
            "Invalid multigranularity_ablation.mode: "
            f"{mode}"
        )


    if mode == "full":
        return mode


    model_cfg = config.get(
        "model",
        {}
    )


    fine_dim = int(
        model_cfg.get(
            "fine_dim",
            32,
        )
    )

    coarse_dim = int(
        model_cfg.get(
            "coarse_dim",
            32,
        )
    )

    output_dim = int(
        model_cfg.get(
            "representation_dim",
            32,
        )
    )


    if (
        fine_dim <= 0
        or coarse_dim <= 0
        or output_dim <= 0
    ):
        raise ValueError(
            "fine_dim, coarse_dim and "
            "representation_dim must be positive"
        )


    device = next(
        model.parameters()
    ).device


    for view_name, encoder in (
        model.view_encoders.items()
    ):

        if mode == "fine_only":

            replacement = _FineOnlyFusion(
                fine_dim=fine_dim,
                output_dim=output_dim,
            )


        elif mode == "coarse_only":

            replacement = _CoarseOnlyFusion(
                coarse_dim=coarse_dim,
                output_dim=output_dim,
            )


        elif mode == "linear_fusion":

            replacement = _LinearFusion(
                fine_dim=fine_dim,
                coarse_dim=coarse_dim,
                output_dim=output_dim,
            )


        replacement = replacement.to(
            device
        )

        encoder.fusion = replacement


    print(
        "MG ablation dimensions:"
    )

    print(
        f"  fine_dim           = {fine_dim}"
    )

    print(
        f"  coarse_dim         = {coarse_dim}"
    )

    print(
        f"  representation_dim = {output_dim}"
    )


    return mode


'''


text = text.replace(
    insert_marker,
    helper_code + insert_marker,
    1,
)


# ============================================================
# 3. Apply ablation immediately after SpaMGCL construction
#    and BEFORE optimizer creation
# ============================================================

old_model_block = '''    model = SpaMGCL(
        input_dims={modality: int(matrix.shape[1]) for modality, matrix in features.items()},
        **model_config,
    ).to(device)
'''


assert old_model_block in text


new_model_block = old_model_block + '''
    mg_ablation_mode = (
        _apply_multigranularity_ablation(
            model,
            config,
        )
    )

    print(
        "Multigranularity ablation mode:",
        mg_ablation_mode,
    )
'''


text = text.replace(
    old_model_block,
    new_model_block,
    1,
)


# ============================================================
# 4. Save runner
# ============================================================

MG_RUNNER.write_text(
    text,
    encoding="utf-8",
)


# Syntax audit
py_compile.compile(
    str(MG_RUNNER),
    doraise=True,
)


print("=" * 100)
print("MG ABLATION RUNNER CREATED")
print("=" * 100)

print(MG_RUNNER)

print(
    "\nPASS: runner compiled successfully."
)

print(
    "PASS: core src/ was NOT modified."
)

MG ABLATION RUNNER CREATED
/kaggle/working/SpaMGCL/SpaMGCL/experiments/run_exp_mg_ablation_400.py

PASS: runner compiled successfully.
PASS: core src/ was NOT modified.


Cell 20：生成 HLN-A1 的 15 个配置

In [28]:
# ============================================================
# Cell 20
# Generate formal HLN-A1 MG-ablation configs
#
# Variants:
#   fine_only
#   coarse_only
#   linear_fusion
#
# Seeds:
#   0-4
#
# Total = 15 configs
# ============================================================

from pathlib import Path
import copy
import yaml


MG_CONFIG_DIR = (
    PROJECT_ROOT
    / "configs"
    / "mg_ablation_400ep"
)

MG_CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


MG_RESULT_ROOT = (
    PROJECT_ROOT
    / "results_mg_ablation_400"
)


BASE_CONFIG = (
    PROJECT_ROOT
    / "configs"
    / "final_clean"
    / "hlna1_clean_200.yaml"
)


assert BASE_CONFIG.exists()


with BASE_CONFIG.open(
    "r",
    encoding="utf-8",
) as f:

    base_cfg = yaml.safe_load(f)


MG_VARIANTS = {
    "fine_only":
        "fine_only",

    "coarse_only":
        "coarse_only",

    "linear_fusion":
        "linear_fusion",
}


created_configs = []


for variant_name, mode in (
    MG_VARIANTS.items()
):

    for seed in range(5):

        cfg = copy.deepcopy(
            base_cfg
        )


        # ====================================================
        # Experiment identity
        # ====================================================

        cfg[
            "experiment"
        ]["seed"] = seed

        cfg[
            "experiment"
        ]["name"] = (
            f"hlna1_mg_"
            f"{variant_name}_"
            f"400ep_seed{seed}"
        )


        # ====================================================
        # Frozen training protocol
        # ====================================================

        cfg[
            "training"
        ]["epochs"] = 400

        cfg[
            "training"
        ]["warm_up_epochs"] = 10


        # ====================================================
        # FULL loss coefficients retained
        # ====================================================

        cfg["loss"]["lambda_rec"] = 1.0
        cfg["loss"]["lambda_mgcl"] = 1.0
        cfg["loss"]["lambda_cluster"] = 0.1
        cfg["loss"]["lambda_spatial"] = 0.0


        # ====================================================
        # MG ablation switch
        # ====================================================

        cfg[
            "multigranularity_ablation"
        ] = {
            "mode": mode
        }


        # ====================================================
        # Frozen official readout
        # ====================================================

        cfg[
            "clustering"
        ]["embedding"] = "concat_z"

        cfg[
            "clustering"
        ]["n_init"] = 20

        cfg[
            "clustering"
        ]["random_state"] = 0


        cfg[
            "refinement"
        ]["enabled"] = True

        cfg[
            "refinement"
        ]["method"] = "bsrr"

        cfg[
            "refinement"
        ]["spatial_k"] = 3


        cfg[
            "evaluation"
        ]["nmi_average_method"] = "max"


        # ====================================================
        # Dedicated output root
        # ====================================================

        cfg[
            "output"
        ]["root"] = (
            "results_mg_ablation_400"
        )


        output_path = (
            MG_CONFIG_DIR
            / (
                f"hlna1_mg_"
                f"{variant_name}_"
                f"400ep_seed{seed}.yaml"
            )
        )


        with output_path.open(
            "w",
            encoding="utf-8",
        ) as f:

            yaml.safe_dump(
                cfg,
                f,
                sort_keys=False,
            )


        created_configs.append(
            output_path
        )


assert len(
    created_configs
) == 15


print("=" * 100)
print("MG ABLATION CONFIG GENERATION")
print("=" * 100)

print(
    f"Created: "
    f"{len(created_configs)} configs"
)


for path in created_configs:
    print(path.name)


print(
    "\nPASS: 15 configs generated."
)

MG ABLATION CONFIG GENERATION
Created: 15 configs
hlna1_mg_fine_only_400ep_seed0.yaml
hlna1_mg_fine_only_400ep_seed1.yaml
hlna1_mg_fine_only_400ep_seed2.yaml
hlna1_mg_fine_only_400ep_seed3.yaml
hlna1_mg_fine_only_400ep_seed4.yaml
hlna1_mg_coarse_only_400ep_seed0.yaml
hlna1_mg_coarse_only_400ep_seed1.yaml
hlna1_mg_coarse_only_400ep_seed2.yaml
hlna1_mg_coarse_only_400ep_seed3.yaml
hlna1_mg_coarse_only_400ep_seed4.yaml
hlna1_mg_linear_fusion_400ep_seed0.yaml
hlna1_mg_linear_fusion_400ep_seed1.yaml
hlna1_mg_linear_fusion_400ep_seed2.yaml
hlna1_mg_linear_fusion_400ep_seed3.yaml
hlna1_mg_linear_fusion_400ep_seed4.yaml

PASS: 15 configs generated.


Cell 21：配置审计

In [29]:
# ============================================================
# Cell 21
# Audit MG-ablation protocol
# ============================================================

import yaml


count = 0


for variant_name, expected_mode in (
    MG_VARIANTS.items()
):

    for seed in range(5):

        path = (
            MG_CONFIG_DIR
            / (
                f"hlna1_mg_"
                f"{variant_name}_"
                f"400ep_seed{seed}.yaml"
            )
        )


        assert path.exists()


        with path.open(
            "r",
            encoding="utf-8",
        ) as f:

            cfg = yaml.safe_load(f)


        # ====================================================
        # Dataset / seed / training
        # ====================================================

        assert (
            cfg["experiment"]["dataset"]
            == "HLN-A1"
        )

        assert (
            int(
                cfg["experiment"]["seed"]
            )
            == seed
        )

        assert (
            int(
                cfg["training"]["epochs"]
            )
            == 400
        )

        assert (
            int(
                cfg["training"][
                    "warm_up_epochs"
                ]
            )
            == 10
        )


        # ====================================================
        # Full loss retained
        # ====================================================

        assert (
            float(
                cfg["loss"]["lambda_rec"]
            )
            == 1.0
        )

        assert (
            float(
                cfg["loss"]["lambda_mgcl"]
            )
            == 1.0
        )

        assert (
            float(
                cfg["loss"]["lambda_cluster"]
            )
            == 0.1
        )

        assert (
            float(
                cfg["loss"]["lambda_spatial"]
            )
            == 0.0
        )


        # ====================================================
        # Ablation mode
        # ====================================================

        actual_mode = (
            cfg[
                "multigranularity_ablation"
            ]["mode"]
        )

        assert (
            actual_mode
            == expected_mode
        )


        # ====================================================
        # Readout
        # ====================================================

        assert (
            cfg[
                "clustering"
            ]["embedding"]
            == "concat_z"
        )

        assert (
            int(
                cfg[
                    "clustering"
                ]["n_init"]
            )
            == 20
        )

        assert (
            int(
                cfg[
                    "clustering"
                ]["random_state"]
            )
            == 0
        )


        assert (
            cfg[
                "refinement"
            ]["enabled"]
            is True
        )

        assert (
            cfg[
                "refinement"
            ]["method"]
            == "bsrr"
        )

        assert (
            int(
                cfg[
                    "refinement"
                ]["spatial_k"]
            )
            == 3
        )


        assert (
            cfg[
                "evaluation"
            ]["nmi_average_method"]
            == "max"
        )


        # ====================================================
        # Dimensions
        # ====================================================

        model_cfg = cfg[
            "model"
        ]

        fine_dim = int(
            model_cfg.get(
                "fine_dim",
                32,
            )
        )

        coarse_dim = int(
            model_cfg.get(
                "coarse_dim",
                32,
            )
        )

        repr_dim = int(
            model_cfg.get(
                "representation_dim",
                32,
            )
        )


        assert fine_dim > 0
        assert coarse_dim > 0
        assert repr_dim > 0


        print(
            f"PASS | "
            f"{variant_name:14s} | "
            f"seed {seed} | "
            f"fine={fine_dim} | "
            f"coarse={coarse_dim} | "
            f"g={repr_dim}"
        )


        count += 1


assert count == 15


print(
    "\n" + "=" * 100
)

print(
    "PASS: MG ABLATION PROTOCOL FROZEN"
)

print(
    "15/15 configs verified."
)

print(
    "=" * 100
)

PASS | fine_only      | seed 0 | fine=32 | coarse=16 | g=16
PASS | fine_only      | seed 1 | fine=32 | coarse=16 | g=16
PASS | fine_only      | seed 2 | fine=32 | coarse=16 | g=16
PASS | fine_only      | seed 3 | fine=32 | coarse=16 | g=16
PASS | fine_only      | seed 4 | fine=32 | coarse=16 | g=16
PASS | coarse_only    | seed 0 | fine=32 | coarse=16 | g=16
PASS | coarse_only    | seed 1 | fine=32 | coarse=16 | g=16
PASS | coarse_only    | seed 2 | fine=32 | coarse=16 | g=16
PASS | coarse_only    | seed 3 | fine=32 | coarse=16 | g=16
PASS | coarse_only    | seed 4 | fine=32 | coarse=16 | g=16
PASS | linear_fusion  | seed 0 | fine=32 | coarse=16 | g=16
PASS | linear_fusion  | seed 1 | fine=32 | coarse=16 | g=16
PASS | linear_fusion  | seed 2 | fine=32 | coarse=16 | g=16
PASS | linear_fusion  | seed 3 | fine=32 | coarse=16 | g=16
PASS | linear_fusion  | seed 4 | fine=32 | coarse=16 | g=16

PASS: MG ABLATION PROTOCOL FROZEN
15/15 configs verified.


Cell 21.5：清理旧失败的 seed0 残留

In [30]:
# ============================================================
# Cell 21.5
# Remove ONLY incomplete old MG seed0 smoke-test directories
#
# Completed runs are NEVER deleted.
# ============================================================

from pathlib import Path
import shutil


for variant in [
    "fine_only",
    "coarse_only",
    "linear_fusion",
]:

    run_dir = (
        PROJECT_ROOT
        / "results_mg_ablation_400"
        / (
            f"hlna1_mg_"
            f"{variant}_"
            f"400ep_seed0"
        )
    )

    if not run_dir.exists():
        print(
            f"NOT FOUND -> clean: {variant}"
        )
        continue


    complete = (
        (run_dir / "metrics.json").exists()
        and
        (run_dir / "pred_labels.npy").exists()
        and
        (run_dir / "config.yaml").exists()
    )


    if complete:

        print(
            f"KEEP completed run: {run_dir}"
        )

    else:

        print(
            f"REMOVE incomplete run: {run_dir}"
        )

        shutil.rmtree(
            run_dir
        )


print(
    "\nPASS: incomplete smoke-test "
    "residuals cleaned safely."
)

REMOVE incomplete run: /kaggle/working/SpaMGCL/SpaMGCL/results_mg_ablation_400/hlna1_mg_fine_only_400ep_seed0
NOT FOUND -> clean: coarse_only
NOT FOUND -> clean: linear_fusion

PASS: incomplete smoke-test residuals cleaned safely.


Cell 22：定义运行函数

In [31]:
# ============================================================
# Cell 22
# Run + audit MG ablation
# ============================================================

from pathlib import Path
import subprocess
import yaml
import json
import numpy as np

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)


def run_mg_ablation(
    config_path,
):

    config_path = Path(
        config_path
    )


    with config_path.open(
        "r",
        encoding="utf-8",
    ) as f:

        cfg = yaml.safe_load(f)


    exp_name = (
        cfg["experiment"]["name"]
    )

    seed = int(
        cfg["experiment"]["seed"]
    )

    mode = (
        cfg[
            "multigranularity_ablation"
        ]["mode"]
    )


    run_dir = (
        PROJECT_ROOT
        / cfg["output"]["root"]
        / exp_name
    )


    print(
        "\n" + "=" * 100
    )

    print(
        f"RUN: {exp_name}"
    )

    print(
        f"Config: {config_path}"
    )

    print(
        f"Output: {run_dir}"
    )

    print(
        "=" * 100
    )


    # ========================================================
    # Existing complete run
    # ========================================================

    complete = (
        run_dir.exists()
        and (
            run_dir
            / "metrics.json"
        ).exists()
        and (
            run_dir
            / "config.yaml"
        ).exists()
        and (
            run_dir
            / "pred_labels.npy"
        ).exists()
        and (
            run_dir
            / "pred_concat_z_kmeans.npy"
        ).exists()
    )


    if complete:

        print(
            "\nExisting completed run "
            "detected -> training skipped."
        )


    else:

        # Never silently overwrite partial results
        if (
            run_dir.exists()
            and any(
                run_dir.iterdir()
            )
        ):

            raise RuntimeError(
                "\nPartial/ambiguous run directory exists:\n"
                f"{run_dir}\n\n"
                "Inspect it before continuing."
            )


        cmd = [
            "python",
            str(MG_RUNNER),
            "--config",
            str(config_path),
        ]


        subprocess.run(
            cmd,
            cwd=PROJECT_ROOT,
            check=True,
        )


    # ========================================================
    # Required outputs
    # ========================================================

    required_files = [
        "metrics.json",
        "config.yaml",
        "gt_labels.npy",
        "pred_labels.npy",
        "pred_concat_z_kmeans.npy",
    ]


    for filename in required_files:

        path = (
            run_dir
            / filename
        )

        assert path.exists(), (
            f"Missing output:\n{path}"
        )


    # ========================================================
    # Independent metric recalculation
    # ========================================================

    gt = np.load(
        run_dir
        / "gt_labels.npy"
    )

    pred = np.load(
        run_dir
        / "pred_labels.npy"
    )

    pred_raw = np.load(
        run_dir
        / "pred_concat_z_kmeans.npy"
    )


    ari = adjusted_rand_score(
        gt,
        pred,
    )

    nmi = (
        normalized_mutual_info_score(
            gt,
            pred,
            average_method="max",
        )
    )


    raw_ari = (
        adjusted_rand_score(
            gt,
            pred_raw,
        )
    )

    raw_nmi = (
        normalized_mutual_info_score(
            gt,
            pred_raw,
            average_method="max",
        )
    )


    # ========================================================
    # Verify saved config
    # ========================================================

    with (
        run_dir
        / "config.yaml"
    ).open(
        "r",
        encoding="utf-8",
    ) as f:

        actual_cfg = yaml.safe_load(f)


    assert (
        actual_cfg[
            "multigranularity_ablation"
        ]["mode"]
        == mode
    )

    assert (
        int(
            actual_cfg[
                "experiment"
            ]["seed"]
        )
        == seed
    )


    # ========================================================
    # Verify metrics.json
    # ========================================================

    with (
        run_dir
        / "metrics.json"
    ).open(
        "r",
        encoding="utf-8",
    ) as f:

        saved_metrics = json.load(f)


    assert np.isclose(
        ari,
        float(
            saved_metrics["ARI"]
        ),
        atol=1e-12,
    )

    assert np.isclose(
        nmi,
        float(
            saved_metrics["NMI"]
        ),
        atol=1e-12,
    )


    # ========================================================
    # Official project audit
    # ========================================================

    audit_script = (
        PROJECT_ROOT
        / "scripts"
        / "audit_run.py"
    )


    audit_status = (
        "NOT FOUND"
    )


    if audit_script.exists():

        audit_proc = subprocess.run(
            [
                "python",
                str(audit_script),
                str(run_dir),
            ],
            cwd=PROJECT_ROOT,
            text=True,
            capture_output=True,
            check=True,
        )

        audit_status = (
            audit_proc.stdout.strip()
        )

        assert (
            "PASS"
            in audit_status.upper()
        ), audit_status


    # ========================================================
    # Report
    # ========================================================

    print(
        "\n" + "-" * 90
    )

    print(
        f"AUDIT PASS | "
        f"HLN-A1 | "
        f"{mode} | "
        f"seed {seed}"
    )

    print(
        "-" * 90
    )

    print(
        f"Official ARI = {ari:.12f}"
    )

    print(
        f"Official NMI = {nmi:.12f}"
    )

    print(
        f"Raw ARI      = {raw_ari:.12f}"
    )

    print(
        f"Raw NMI      = {raw_nmi:.12f}"
    )

    print(
        f"BSRR ΔARI    = "
        f"{ari - raw_ari:+.12f}"
    )

    print(
        f"BSRR ΔNMI    = "
        f"{nmi - raw_nmi:+.12f}"
    )

    print(
        f"\nMG mode      = {mode}"
    )

    print(
        "Official audit_run.py:",
        (
            "PASS"
            if "PASS" in audit_status.upper()
            else audit_status
        )
    )


    return {
        "variant":
            mode,

        "seed":
            seed,

        "ARI":
            ari,

        "NMI":
            nmi,

        "raw_ARI":
            raw_ari,

        "raw_NMI":
            raw_nmi,

        "run_dir":
            run_dir,
    }


print(
    "PASS: MG ablation execution "
    "helper ready."
)

PASS: MG ablation execution helper ready.


Cell 23：先跑 3 个 seed0 smoke test

In [32]:
# ============================================================
# Cell 23
# HLN-A1 MG ablation smoke tests
#
# Run ONLY seed0:
#   fine_only
#   coarse_only
#   linear_fusion
#
# STOP after this cell.
# ============================================================

FULL_SEED0_ARI = (
    0.220614
)

FULL_SEED0_NMI = (
    0.358325
)


SMOKE_RESULTS = {}


for variant in [
    "fine_only",
    "coarse_only",
    "linear_fusion",
]:

    config_path = (
        MG_CONFIG_DIR
        / (
            f"hlna1_mg_"
            f"{variant}_"
            f"400ep_seed0.yaml"
        )
    )


    result = run_mg_ablation(
        config_path
    )


    SMOKE_RESULTS[
        variant
    ] = result


# ============================================================
# Comparison
# ============================================================

print(
    "\n" + "=" * 100
)

print(
    "HLN-A1 MG ABLATION "
    "SEED0 SMOKE SUMMARY"
)

print(
    "=" * 100
)


print(
    "\nFull"
)

print(
    f"  ARI = "
    f"{FULL_SEED0_ARI:.6f}"
)

print(
    f"  NMI = "
    f"{FULL_SEED0_NMI:.6f}"
)


for variant in [
    "fine_only",
    "coarse_only",
    "linear_fusion",
]:

    result = (
        SMOKE_RESULTS[
            variant
        ]
    )


    print(
        f"\n{variant}"
    )

    print(
        f"  ARI  = "
        f"{result['ARI']:.6f}"
    )

    print(
        f"  NMI  = "
        f"{result['NMI']:.6f}"
    )

    print(
        f"  ΔARI = "
        f"{result['ARI'] - FULL_SEED0_ARI:+.6f}"
    )

    print(
        f"  ΔNMI = "
        f"{result['NMI'] - FULL_SEED0_NMI:+.6f}"
    )


print(
    "\n" + "=" * 100
)

print(
    "STOP HERE."
)

print(
    "Do NOT run seeds 1-4 yet."
)

print(
    "=" * 100
)


RUN: hlna1_mg_fine_only_400ep_seed0
Config: /kaggle/working/SpaMGCL/SpaMGCL/configs/mg_ablation_400ep/hlna1_mg_fine_only_400ep_seed0.yaml
Output: /kaggle/working/SpaMGCL/SpaMGCL/results_mg_ablation_400/hlna1_mg_fine_only_400ep_seed0


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


MG ablation dimensions:
  fine_dim           = 32
  coarse_dim         = 16
  representation_dim = 16
Multigranularity ablation mode: fine_only
Dataset: HLN-A1 | spots=3484 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3484, 3484), nnz=13206
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=8.417746 | rec=0.252383 | mgcl=8.165362 | cluster=2.951158 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=5.375e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.950 | gradC=0.000e+00
epoch 002/400 | total=8.404860 | rec=0.242660 | mgcl=8.162200 | cluster=2.950328 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=6.983e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.956 | gradC=0.000e+00
epoch 003/400 | total=8.392852 | rec=0.233263 | mgcl=8.159589 | cluster=2.949564 | spatial_loss=0.000000 | lambda_spatial=0 | sc_ena

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


MG ablation dimensions:
  fine_dim           = 32
  coarse_dim         = 16
  representation_dim = 16
Multigranularity ablation mode: coarse_only
Dataset: HLN-A1 | spots=3484 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3484, 3484), nnz=13206
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=8.408672 | rec=0.252383 | mgcl=8.156289 | cluster=2.951158 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=4.858e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.950 | gradC=0.000e+00
epoch 002/400 | total=8.398360 | rec=0.242176 | mgcl=8.156184 | cluster=2.950272 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=6.517e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.956 | gradC=0.000e+00
epoch 003/400 | total=8.388483 | rec=0.232376 | mgcl=8.156108 | cluster=2.949481 | spatial_loss=0.000000 | lambda_spatial=0 | sc_e

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


MG ablation dimensions:
  fine_dim           = 32
  coarse_dim         = 16
  representation_dim = 16
Multigranularity ablation mode: linear_fusion
Dataset: HLN-A1 | spots=3484 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3484, 3484), nnz=13206
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=8.418587 | rec=0.252383 | mgcl=8.166203 | cluster=2.951158 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=6.138e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.950 | gradC=0.000e+00
epoch 002/400 | total=8.404719 | rec=0.242752 | mgcl=8.161967 | cluster=2.950305 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=7.946e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.956 | gradC=0.000e+00
epoch 003/400 | total=8.392299 | rec=0.233431 | mgcl=8.158868 | cluster=2.949525 | spatial_loss=0.000000 | lambda_spatial=0 | sc

Cell 24：跑剩余 seeds 1–4

In [33]:
# ============================================================
# Cell 24
# Run remaining formal MG ablations
#
# Already completed:
#   seed0 for all three variants
#
# Now run:
#   seeds 1-4
#
# Total new runs = 12
# ============================================================

FORMAL_MG_RESULTS = []


for variant in [
    "fine_only",
    "coarse_only",
    "linear_fusion",
]:

    for seed in range(1, 5):

        config_path = (
            MG_CONFIG_DIR
            / (
                f"hlna1_mg_"
                f"{variant}_"
                f"400ep_seed{seed}.yaml"
            )
        )


        result = run_mg_ablation(
            config_path
        )


        FORMAL_MG_RESULTS.append(
            result
        )


print(
    "\n" + "=" * 100
)

print(
    "PASS: REMAINING MG ABLATION RUNS COMPLETE"
)

print(
    f"Completed new runs: "
    f"{len(FORMAL_MG_RESULTS)}/12"
)

print(
    "=" * 100
)


RUN: hlna1_mg_fine_only_400ep_seed1
Config: /kaggle/working/SpaMGCL/SpaMGCL/configs/mg_ablation_400ep/hlna1_mg_fine_only_400ep_seed1.yaml
Output: /kaggle/working/SpaMGCL/SpaMGCL/results_mg_ablation_400/hlna1_mg_fine_only_400ep_seed1


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


MG ablation dimensions:
  fine_dim           = 32
  coarse_dim         = 16
  representation_dim = 16
Multigranularity ablation mode: fine_only
Dataset: HLN-A1 | spots=3484 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3484, 3484), nnz=13206
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=8.405725 | rec=0.237463 | mgcl=8.168262 | cluster=2.950006 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=9.404e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.959 | gradC=0.000e+00
epoch 002/400 | total=8.394179 | rec=0.229149 | mgcl=8.165030 | cluster=2.949148 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=7.021e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.966 | gradC=0.000e+00
epoch 003/400 | total=8.383529 | rec=0.221129 | mgcl=8.162400 | cluster=2.948393 | spatial_loss=0.000000 | lambda_spatial=0 | sc_ena

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


MG ablation dimensions:
  fine_dim           = 32
  coarse_dim         = 16
  representation_dim = 16
Multigranularity ablation mode: fine_only
Dataset: HLN-A1 | spots=3484 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3484, 3484), nnz=13206
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=8.417406 | rec=0.254117 | mgcl=8.163289 | cluster=2.951155 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.445e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.950 | gradC=0.000e+00
epoch 002/400 | total=8.405591 | rec=0.244694 | mgcl=8.160897 | cluster=2.950474 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=9.066e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.955 | gradC=0.000e+00
epoch 003/400 | total=8.394578 | rec=0.235680 | mgcl=8.158898 | cluster=2.949812 | spatial_loss=0.000000 | lambda_spatial=0 | sc_ena

...

MG ablation dimensions:
  fine_dim           = 32
  coarse_dim         = 16
  representation_dim = 16
Multigranularity ablation mode: linear_fusion
Dataset: HLN-A1 | spots=3484 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3484, 3484), nnz=13206
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=8.421639 | rec=0.254117 | mgcl=8.167522 | cluster=2.951155 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=7.783e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.950 | gradC=0.000e+00
epoch 002/400 | total=8.408360 | rec=0.244813 | mgcl=8.163547 | cluster=2.950453 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.774e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.955 | gradC=0.000e+00
epoch 003/400 | total=8.396519 | rec=0.235864 | mgcl=8.160655 | cluster=2.949780 | spatial_loss=0.000000 | lambda_spatial=0 | sc

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


MG ablation dimensions:
  fine_dim           = 32
  coarse_dim         = 16
  representation_dim = 16
Multigranularity ablation mode: linear_fusion
Dataset: HLN-A1 | spots=3484 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3484, 3484), nnz=13206
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=8.428452 | rec=0.261168 | mgcl=8.167285 | cluster=2.948812 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.059e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.962 | gradC=0.000e+00
epoch 002/400 | total=8.415508 | rec=0.251724 | mgcl=8.163785 | cluster=2.948255 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=5.973e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.967 | gradC=0.000e+00
epoch 003/400 | total=8.403708 | rec=0.242542 | mgcl=8.161166 | cluster=2.947728 | spatial_loss=0.000000 | lambda_spatial=0 | sc

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


MG ablation dimensions:
  fine_dim           = 32
  coarse_dim         = 16
  representation_dim = 16
Multigranularity ablation mode: linear_fusion
Dataset: HLN-A1 | spots=3484 | device=cuda
Modalities: RNA, ADT | label=Spatial_Label
Spatial graph: shape=(3484, 3484), nnz=13206
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/400 | total=8.427704 | rec=0.265092 | mgcl=8.162612 | cluster=2.950062 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=3.357e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.953 | gradC=0.000e+00
epoch 002/400 | total=8.414253 | rec=0.255515 | mgcl=8.158739 | cluster=2.949081 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=4.398e-03 | neg_count=12134772 | snf_masked_positions=0 | effC=9.962 | gradC=0.000e+00
epoch 003/400 | total=8.401987 | rec=0.246320 | mgcl=8.155666 | cluster=2.948233 | spatial_loss=0.000000 | lambda_spatial=0 | sc

Cell 25：汇总三个配置的 5-seed 结果

In [34]:
# ============================================================
# Cell 25
# Collect and summarize 5-seed MG ablation results
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)


FULL_REF = {
    0: (0.220614, 0.358325),
    1: (0.273804, 0.393244),
    2: (0.278258, 0.404178),
    3: (0.237522, 0.353609),
    4: (0.263325, 0.387647),
}


rows = []


# ------------------------------------------------------------
# Full
# ------------------------------------------------------------

for seed in range(5):

    ari, nmi = FULL_REF[seed]

    rows.append(
        {
            "variant": "Full",
            "seed": seed,
            "ARI": ari,
            "NMI": nmi,
        }
    )


# ------------------------------------------------------------
# MG ablations
# ------------------------------------------------------------

for variant in [
    "fine_only",
    "coarse_only",
    "linear_fusion",
]:

    for seed in range(5):

        run_dir = (
            PROJECT_ROOT
            / "results_mg_ablation_400"
            / (
                f"hlna1_mg_"
                f"{variant}_"
                f"400ep_seed{seed}"
            )
        )

        assert run_dir.exists(), run_dir


        gt = np.load(
            run_dir
            / "gt_labels.npy"
        )

        pred = np.load(
            run_dir
            / "pred_labels.npy"
        )


        ari = adjusted_rand_score(
            gt,
            pred,
        )

        nmi = normalized_mutual_info_score(
            gt,
            pred,
            average_method="max",
        )


        rows.append(
            {
                "variant": variant,
                "seed": seed,
                "ARI": ari,
                "NMI": nmi,
            }
        )


df = pd.DataFrame(rows)


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

summary = (
    df
    .groupby(
        "variant",
        sort=False,
    )
    .agg(
        ARI_mean=("ARI", "mean"),
        ARI_std=(
            "ARI",
            lambda x:
                np.std(
                    x,
                    ddof=0,
                )
        ),
        NMI_mean=("NMI", "mean"),
        NMI_std=(
            "NMI",
            lambda x:
                np.std(
                    x,
                    ddof=0,
                )
        ),
    )
    .reset_index()
)


full_ari = (
    summary.loc[
        summary["variant"] == "Full",
        "ARI_mean",
    ].iloc[0]
)

full_nmi = (
    summary.loc[
        summary["variant"] == "Full",
        "NMI_mean",
    ].iloc[0]
)


summary["Delta_ARI_vs_Full"] = (
    summary["ARI_mean"]
    - full_ari
)

summary["Delta_NMI_vs_Full"] = (
    summary["NMI_mean"]
    - full_nmi
)


summary["ARI"] = summary.apply(
    lambda r:
        f"{r['ARI_mean']:.6f} "
        f"± {r['ARI_std']:.6f}",
    axis=1,
)

summary["NMI"] = summary.apply(
    lambda r:
        f"{r['NMI_mean']:.6f} "
        f"± {r['NMI_std']:.6f}",
    axis=1,
)


print(
    "\n" + "=" * 110
)

print(
    "HLN-A1 MULTIGRANULARITY ABLATION"
)

print(
    "5 seeds | 400 epochs | mean ± std"
)

print(
    "=" * 110
)


print(
    summary[
        [
            "variant",
            "ARI",
            "NMI",
            "Delta_ARI_vs_Full",
            "Delta_NMI_vs_Full",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:+.6f}",
    )
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

OUT_DIR = (
    PROJECT_ROOT
    / "ablation_summary_400"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


df.to_csv(
    OUT_DIR
    / "HLNA1_mg_ablation_5seeds_raw.csv",
    index=False,
)


summary.to_csv(
    OUT_DIR
    / "HLNA1_mg_ablation_5seeds_summary.csv",
    index=False,
)


print(
    "\nSaved MG ablation summary."
)


HLN-A1 MULTIGRANULARITY ABLATION
5 seeds | 400 epochs | mean ± std
      variant                 ARI                 NMI  Delta_ARI_vs_Full  Delta_NMI_vs_Full
         Full 0.254705 ± 0.022142 0.379401 ± 0.019915          +0.000000          +0.000000
    fine_only 0.236880 ± 0.024257 0.351806 ± 0.009865          -0.017825          -0.027594
  coarse_only 0.236784 ± 0.012546 0.360722 ± 0.015366          -0.017921          -0.018679
linear_fusion 0.243713 ± 0.024313 0.364073 ± 0.018845          -0.010992          -0.015328

Saved MG ablation summary.


Cell 26：生成最终 Table 3

In [35]:
# ============================================================
# Cell 26
# Final HLN-A1 ablation Table 3
#
# Main-paper variants:
#   Full
#   w/o Sample CL
#   Fine-only
#   Coarse-only
#   w/o nonlinear fusion
#
# 5 seeds | 400 epochs | mean ± std
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path


FINAL_ABLATION_ROWS = [
    {
        "Variant": "Full",
        "ARI_mean": 0.2547046,
        "ARI_std": 0.0221416882,
        "NMI_mean": 0.3794006,
        "NMI_std": 0.0199145672,
    },

    {
        "Variant": "w/o Sample CL",
        "ARI_mean": 0.195045844775,
        "ARI_std": 0.02064039599,
        "NMI_mean": 0.3083205490014,
        "NMI_std": 0.02137697526,
    },

    {
        "Variant": "Fine-only",
        "ARI_mean": 0.236880,
        "ARI_std": 0.024257,
        "NMI_mean": 0.351806,
        "NMI_std": 0.009865,
    },

    {
        "Variant": "Coarse-only",
        "ARI_mean": 0.236784,
        "ARI_std": 0.012546,
        "NMI_mean": 0.360722,
        "NMI_std": 0.015366,
    },

    {
        "Variant": "w/o nonlinear fusion",
        "ARI_mean": 0.243713,
        "ARI_std": 0.024313,
        "NMI_mean": 0.364073,
        "NMI_std": 0.018845,
    },
]


table3 = pd.DataFrame(
    FINAL_ABLATION_ROWS
)


full_ari = (
    table3.loc[
        table3["Variant"] == "Full",
        "ARI_mean",
    ].iloc[0]
)

full_nmi = (
    table3.loc[
        table3["Variant"] == "Full",
        "NMI_mean",
    ].iloc[0]
)


table3["Delta_ARI_vs_Full"] = (
    table3["ARI_mean"]
    - full_ari
)

table3["Delta_NMI_vs_Full"] = (
    table3["NMI_mean"]
    - full_nmi
)


table3["ARI"] = table3.apply(
    lambda r:
        f"{r['ARI_mean']:.6f} "
        f"± {r['ARI_std']:.6f}",
    axis=1,
)


table3["NMI"] = table3.apply(
    lambda r:
        f"{r['NMI_mean']:.6f} "
        f"± {r['NMI_std']:.6f}",
    axis=1,
)


print(
    "=" * 110
)

print(
    "FINAL TABLE 3 — HLN-A1 ABLATION STUDY"
)

print(
    "400 epochs | seeds 0-4 | mean ± std"
)

print(
    "=" * 110
)


print(
    table3[
        [
            "Variant",
            "ARI",
            "NMI",
            "Delta_ARI_vs_Full",
            "Delta_NMI_vs_Full",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:+.6f}",
    )
)


OUT_DIR = (
    PROJECT_ROOT
    / "ablation_summary_400"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


table3.to_csv(
    OUT_DIR
    / "Table3_HLNA1_final_ablation_5seeds.csv",
    index=False,
)


print(
    "\nSaved:"
)

print(
    OUT_DIR
    / "Table3_HLNA1_final_ablation_5seeds.csv"
)

FINAL TABLE 3 — HLN-A1 ABLATION STUDY
400 epochs | seeds 0-4 | mean ± std
             Variant                 ARI                 NMI  Delta_ARI_vs_Full  Delta_NMI_vs_Full
                Full 0.254705 ± 0.022142 0.379401 ± 0.019915          +0.000000          +0.000000
       w/o Sample CL 0.195046 ± 0.020640 0.308321 ± 0.021377          -0.059659          -0.071080
           Fine-only 0.236880 ± 0.024257 0.351806 ± 0.009865          -0.017825          -0.027595
         Coarse-only 0.236784 ± 0.012546 0.360722 ± 0.015366          -0.017921          -0.018679
w/o nonlinear fusion 0.243713 ± 0.024313 0.364073 ± 0.018845          -0.010992          -0.015328

Saved:
/kaggle/working/SpaMGCL/SpaMGCL/ablation_summary_400/Table3_HLNA1_final_ablation_5seeds.csv


Cell 27：从真实 raw 数据重新生成最终表

In [36]:
# ============================================================
# Cell 27
# Rebuild FINAL Table 3 directly from raw run-level results
#
# Avoid manual rounding / transcription.
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path


OUT_DIR = (
    PROJECT_ROOT
    / "ablation_summary_400"
)


# ============================================================
# 1. Current original ablation raw
#    Contains:
#      Full
#      w/o Sample CL
#      w/o Cluster CL
# ============================================================

original_raw_path = (
    OUT_DIR
    / "HLNA1_ablation_current_5seeds_raw.csv"
)

assert original_raw_path.exists()

original_raw = pd.read_csv(
    original_raw_path
)


# Keep only main-paper rows
original_main = (
    original_raw[
        original_raw["variant"].isin(
            [
                "Full",
                "w/o Sample CL",
            ]
        )
    ][
        [
            "variant",
            "seed",
            "ARI",
            "NMI",
        ]
    ]
    .copy()
)


# ============================================================
# 2. Multigranularity raw
# ============================================================

mg_raw_path = (
    OUT_DIR
    / "HLNA1_mg_ablation_5seeds_raw.csv"
)

assert mg_raw_path.exists()

mg_raw = pd.read_csv(
    mg_raw_path
)


mg_main = (
    mg_raw[
        mg_raw["variant"].isin(
            [
                "fine_only",
                "coarse_only",
                "linear_fusion",
            ]
        )
    ].copy()
)


# Paper-facing names
mg_main["variant"] = (
    mg_main["variant"]
    .replace(
        {
            "fine_only":
                "Fine-only",

            "coarse_only":
                "Coarse-only",

            "linear_fusion":
                "w/o nonlinear fusion",
        }
    )
)


# ============================================================
# 3. Merge
# ============================================================

final_raw = pd.concat(
    [
        original_main,
        mg_main[
            [
                "variant",
                "seed",
                "ARI",
                "NMI",
            ]
        ],
    ],
    ignore_index=True,
)


final_raw = final_raw.rename(
    columns={
        "variant": "Variant",
        "seed": "Seed",
    }
)


VARIANT_ORDER = [
    "Full",
    "w/o Sample CL",
    "Fine-only",
    "Coarse-only",
    "w/o nonlinear fusion",
]


final_raw["Variant"] = pd.Categorical(
    final_raw["Variant"],
    categories=VARIANT_ORDER,
    ordered=True,
)


final_raw = (
    final_raw
    .sort_values(
        [
            "Variant",
            "Seed",
        ]
    )
    .reset_index(
        drop=True
    )
)


# Each variant MUST have exactly 5 seeds
counts = (
    final_raw
    .groupby(
        "Variant",
        observed=True,
    )
    .size()
)


print(
    "Run counts:"
)

print(counts)


assert (
    counts == 5
).all()


# ============================================================
# 4. Mean ± std
# ============================================================

summary_rows = []


for variant in VARIANT_ORDER:

    d = (
        final_raw[
            final_raw["Variant"]
            == variant
        ]
        .sort_values("Seed")
    )


    ari = d["ARI"].to_numpy()
    nmi = d["NMI"].to_numpy()


    summary_rows.append(
        {
            "Variant": variant,

            "ARI_mean":
                ari.mean(),

            "ARI_std":
                ari.std(
                    ddof=0
                ),

            "NMI_mean":
                nmi.mean(),

            "NMI_std":
                nmi.std(
                    ddof=0
                ),
        }
    )


final_table = pd.DataFrame(
    summary_rows
)


full_ari = (
    final_table.loc[
        final_table["Variant"] == "Full",
        "ARI_mean",
    ].iloc[0]
)

full_nmi = (
    final_table.loc[
        final_table["Variant"] == "Full",
        "NMI_mean",
    ].iloc[0]
)


final_table["Delta_ARI_vs_Full"] = (
    final_table["ARI_mean"]
    - full_ari
)

final_table["Delta_NMI_vs_Full"] = (
    final_table["NMI_mean"]
    - full_nmi
)


final_table["ARI"] = final_table.apply(
    lambda r:
        f"{r['ARI_mean']:.6f} "
        f"± {r['ARI_std']:.6f}",
    axis=1,
)


final_table["NMI"] = final_table.apply(
    lambda r:
        f"{r['NMI_mean']:.6f} "
        f"± {r['NMI_std']:.6f}",
    axis=1,
)


# ============================================================
# 5. Display
# ============================================================

print(
    "\n" + "=" * 115
)

print(
    "FINAL VERIFIED TABLE 3"
)

print(
    "HLN-A1 | 400 epochs | seeds 0-4"
)

print(
    "=" * 115
)


print(
    final_table[
        [
            "Variant",
            "ARI",
            "NMI",
            "Delta_ARI_vs_Full",
            "Delta_NMI_vs_Full",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:+.6f}",
    )
)


# ============================================================
# 6. Save
# ============================================================

final_raw.to_csv(
    OUT_DIR
    / "Table3_HLNA1_final_ablation_RAW.csv",
    index=False,
)


final_table.to_csv(
    OUT_DIR
    / "Table3_HLNA1_final_ablation_VERIFIED.csv",
    index=False,
)


print(
    "\nSaved:"
)

print(
    OUT_DIR
    / "Table3_HLNA1_final_ablation_RAW.csv"
)

print(
    OUT_DIR
    / "Table3_HLNA1_final_ablation_VERIFIED.csv"
)

print(
    "\nPASS: FINAL TABLE 3 VERIFIED FROM RAW RESULTS."
)

Run counts:
Variant
Full                    5
w/o Sample CL           5
Fine-only               5
Coarse-only             5
w/o nonlinear fusion    5
dtype: int64

FINAL VERIFIED TABLE 3
HLN-A1 | 400 epochs | seeds 0-4
             Variant                 ARI                 NMI  Delta_ARI_vs_Full  Delta_NMI_vs_Full
                Full 0.254705 ± 0.022142 0.379401 ± 0.019915          +0.000000          +0.000000
       w/o Sample CL 0.195046 ± 0.020640 0.308321 ± 0.021377          -0.059659          -0.071080
           Fine-only 0.236880 ± 0.024257 0.351806 ± 0.009865          -0.017825          -0.027594
         Coarse-only 0.236784 ± 0.012546 0.360722 ± 0.015366          -0.017921          -0.018679
w/o nonlinear fusion 0.243713 ± 0.024313 0.364073 ± 0.018845          -0.010992          -0.015328

Saved:
/kaggle/working/SpaMGCL/SpaMGCL/ablation_summary_400/Table3_HLNA1_final_ablation_RAW.csv
/kaggle/working/SpaMGCL/SpaMGCL/ablation_summary_400/Table3_HLNA1_final_ablation_VERIFIED.

In [37]:
# ============================================================
# FINAL CELL
# Archive current HLN-A1 ablation work before closing Kaggle
# ============================================================

from pathlib import Path
import shutil

ARCHIVE_DIR = (
    PROJECT_ROOT
    / "HLNA1_ablation_400_FINAL"
)

if ARCHIVE_DIR.exists():
    shutil.rmtree(ARCHIVE_DIR)

ARCHIVE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# Copy summary tables
# ------------------------------------------------------------

src = PROJECT_ROOT / "ablation_summary_400"

if src.exists():
    shutil.copytree(
        src,
        ARCHIVE_DIR / "ablation_summary_400",
    )


# ------------------------------------------------------------
# Copy MG configs
# ------------------------------------------------------------

src = (
    PROJECT_ROOT
    / "configs"
    / "mg_ablation_400ep"
)

if src.exists():
    shutil.copytree(
        src,
        ARCHIVE_DIR
        / "configs"
        / "mg_ablation_400ep",
    )


# ------------------------------------------------------------
# Copy essential evidence from MG runs
# Avoid large checkpoints / embeddings
# ------------------------------------------------------------

run_root = (
    PROJECT_ROOT
    / "results_mg_ablation_400"
)

dst_root = (
    ARCHIVE_DIR
    / "results_mg_ablation_400"
)

ESSENTIAL = [
    "metrics.json",
    "config.yaml",
    "manifest.json",
    "gt_labels.npy",
    "pred_labels.npy",
    "pred_concat_z_kmeans.npy",
]


if run_root.exists():

    for run_dir in sorted(
        run_root.iterdir()
    ):

        if not run_dir.is_dir():
            continue

        out = (
            dst_root
            / run_dir.name
        )

        out.mkdir(
            parents=True,
            exist_ok=True,
        )

        for name in ESSENTIAL:

            src_file = (
                run_dir
                / name
            )

            if src_file.exists():

                shutil.copy2(
                    src_file,
                    out / name,
                )


# ------------------------------------------------------------
# ZIP
# ------------------------------------------------------------

zip_path = shutil.make_archive(
    str(
        PROJECT_ROOT
        / "HLNA1_ablation_400_FINAL"
    ),
    "zip",
    root_dir=ARCHIVE_DIR,
)


print("=" * 100)
print("FINAL ABLATION ARCHIVE READY")
print("=" * 100)
print(zip_path)
print("\nYou can close the Jupyter/Kaggle session after saving/downloading this archive.")

FINAL ABLATION ARCHIVE READY
/kaggle/working/SpaMGCL/SpaMGCL/HLNA1_ablation_400_FINAL.zip

You can close the Jupyter/Kaggle session after saving/downloading this archive.
